# Notebook 02 (V4_3) — $\nu_b=\xi_W=0$ HKT Tail and Bubble-Share Bounds

This replicates `Two_country_proudction_fixed_nu_b/02_v9_ahp_hkt_scalar_tail_bubble_bounds_fixed_nu_b.ipynb`;
its upstream source is common-growth Notebook 17. The zero case retains
the selected AHP-matching calibration `pathwise_hi_exp_pi075` for every other
primitive and sets
$$
\nu_b=\xi_W=0.
$$
The absorbing nonlinear system remains seven-dimensional, with no endogenous
or switch-specific exponent. Since $G_{N,US}^{0}=G_{N,W}^{0}=1$, the
common-growth equality is an identity and is omitted from the solver and
validity code.

The notebook:

1. reads the source AHP artifacts only as immutable calibration/comparison data;
2. searches the zero-exponent no-buffer continuation frontier;
3. verifies that one normalized absorbing equilibrium is reused with
   $q_i,d_i\propto1/N_i$ and constant policies/aggregates;
4. applies the independent HKT scalar-entry test and terminal-rule diagnostics;
5. computes fundamental-value-free bubble-share bounds.

Model validity is based on equilibrium residuals, feasibility, accounting,
primitive exponent ordering, the extended algebraically restated absorbing
sequence, and the normalization invariants. Equality to the source AHP path is
a comparison diagnostic, not a replication gate.


## 0. Setup

Run all cells from a fresh Julia kernel. Set `AHP_ZERO_NU_B_TAIL_MODE=smoke` for
a short execution check. Replication mode searches the documented grid through
`AHP_ZERO_NU_B_SEARCH_CAP` (default 81), which brackets the observed zero-case
frontier; `AHP_ZERO_NU_B_T_CANDIDATES` overrides the grid.

`AHP_ZERO_NU_B_BRANCH_ITER_SCHEDULE` controls escalating all-$u$ branch budgets
(default `500,1000`). The absorbing equilibrium itself is solved only once per
parameterization. All outputs are isolated under the zero-exponent directory.


In [ ]:
using Pkg

const ZERO_NU_B_DIRNAME = "Two_country_proudction_zero_nu_b"
const COMMON_GROWTH_SOURCE_DIRNAME = "Two_country_production_common_growth"

function find_project_dir(start_dir=pwd())
    dir = abspath(start_dir)
    while true
        for candidate in (dir, joinpath(dir, "Codes"))
            project = joinpath(candidate, "Project.toml")
            model = joinpath(candidate, ZERO_NU_B_DIRNAME, "TwoCountryProductionOLG.jl")
            isfile(project) && isfile(model) && return candidate
        end
        parent = dirname(dir)
        parent == dir &&
            error("Could not locate Codes/Project.toml and the zero-nu_b model from $(start_dir)")
        dir = parent
    end
end

PROJECT_DIR = find_project_dir()
Pkg.activate(PROJECT_DIR)
cd(PROJECT_DIR)

MODEL_FILE = joinpath(PROJECT_DIR, ZERO_NU_B_DIRNAME, "TwoCountryProductionOLG.jl")
if @isdefined ProductionParams
    @isdefined(_require_zero_nu_b) || error(
        "A non-zero-exponent TwoCountryProductionOLG solver is already loaded. " *
        "Restart the Julia kernel before running Notebook 02.")
else
    include(MODEL_FILE)
end

using DelimitedFiles, SHA
using Markdown, Printf, Statistics, LinearAlgebra, Dates
using Plots, LaTeXStrings
using Plots.PlotMeasures

gr()
default(size=(940, 560), framestyle=:box, grid=:y, legend=:best,
        fontfamily="Computer Modern", linewidth=2,
        titlefontsize=11, guidefontsize=10, tickfontsize=9, legendfontsize=8,
        left_margin=8mm, right_margin=6mm, top_margin=7mm, bottom_margin=8mm)

DEFAULT_OUTDIR = joinpath(PROJECT_DIR, ZERO_NU_B_DIRNAME, "outputs_zero_nu_b",
                          "02_ahp_hkt_scalar_tail_bubble_bounds")
OUTDIR = get(ENV, "AHP_ZERO_NU_B_TAIL_OUTDIR", DEFAULT_OUTDIR)
mkpath(OUTDIR)

RUN_MODE = Symbol(get(ENV, "AHP_ZERO_NU_B_TAIL_MODE", "replication"))
@assert RUN_MODE in (:replication, :grid, :smoke) "RUN_MODE must be :replication, :grid, or :smoke"
RUN_STARTED_UTC = Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SSZ")

println("Project directory:            ", PROJECT_DIR)
println("Zero-nu_b model:             ", MODEL_FILE)
println("Output directory:             ", OUTDIR)
println("Run mode:                     ", RUN_MODE)
println("Zero nu_b = xi_W:            0.0")
println("Common-growth equation:       absent (1 = 1 identity)")


## 1. AHP Calibration, Zero-Exponent Override, Search Grid, and Tolerances

Every source AHP primitive is preserved except the explicit absorbing overrides
$\nu_b=\xi_W=0$. The source values of those two exponents are provenance only;
they are not checked against the zero-exponent factory and never enter the
solver.


In [ ]:
RESID_TOL = 1e-5
PSI_INTERIOR_TOL = 0.02
EQUITY_INTERIOR_TOL = 0.01
THETA_INTERIOR_TOL = 0.01
PHI_INTERIOR_MARGIN = 1e-8
ACCOUNTING_SCALED_TOL = 1e-7
ABSORBING_INVARIANCE_TOL = 1e-10
CERT_TOL = RESID_TOL
COLD_PATH_TOL = 1e-5
NO_BUFFER = 0

function nb17_read_csv_any(path::AbstractString)
    isfile(path) || error("Required CSV not found: $(path)")
    data, header = readdlm(path, ',', Any, '\n'; header=true, quotes=true)
    matrix = ndims(data) == 1 ? reshape(data, 1, :) : data
    names = String.(vec(header))
    size(matrix, 2) == length(names) || error("CSV header mismatch: $(path)")
    return matrix, names
end

function nb17_column_index(names, name)
    index = findfirst(==(name), names)
    index === nothing && error("Missing CSV column: $(name)")
    return index
end

nb17_as_float(x) = x isa Number ? Float64(x) : parse(Float64, string(x))
nb17_as_int(x) = x isa Integer ? Int(x) : Int(round(nb17_as_float(x)))
nb17_as_bool(x) = lowercase(string(x)) == "true"

NB15_RESULT_DIR = joinpath(PROJECT_DIR, COMMON_GROWTH_SOURCE_DIRNAME, "outputs_v43",
                           "ahp_pattern_first_window_common_growth_search")
NB15_CANDIDATE_CSV = joinpath(NB15_RESULT_DIR, "candidate_summary.csv")
NB15_TRACKED_PATH_CSV = joinpath(NB15_RESULT_DIR, "best_ahp_decomposition.csv")

candidate_data, candidate_names = nb17_read_csv_any(NB15_CANDIDATE_CSV)
horizon_col = nb17_column_index(candidate_names, "horizon")
valid_col = nb17_column_index(candidate_names, "hard_valid")
score_col = nb17_column_index(candidate_names, "score")
final_indices = [i for i in axes(candidate_data, 1)
                 if nb17_as_int(candidate_data[i, horizon_col]) == 30 &&
                    nb17_as_bool(candidate_data[i, valid_col])]
isempty(final_indices) && error("Notebook 15 has no hard-valid T=30 finalist")
selected_index = final_indices[argmin([nb17_as_float(candidate_data[i, score_col])
                                      for i in final_indices])]
AHP_SELECTED_ROW = Dict(candidate_names[j] => candidate_data[selected_index, j]
                        for j in eachindex(candidate_names))
AHP_SELECTED_LABEL = string(AHP_SELECTED_ROW["label"])
AHP_SELECTED_LABEL == "pathwise_hi_exp_pi075" ||
    error("Unexpected Notebook 15 winner: $(AHP_SELECTED_LABEL)")
AHP_EXPECTED_T15_REBASED_NFA = nb17_as_float(AHP_SELECTED_ROW["target_rebased_NFA"])
AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE = nb17_as_float(AHP_SELECTED_ROW["nu_b_calibrated"])
const ZERO_NU_B = 0.0
AHP_SELECTED_CLASSIFICATION = string(AHP_SELECTED_ROW["classification"])

const AHP_CALIBRATION = (
    β=0.45, γ=0.25, π_persist=0.75,
    a_US=0.20, ϑ_US=0.85,
    a_W=0.06, H_W=3.0, L_W=3.0,
    A_X_US_u=15.0, A_L_US_u=1.5,
    ν_b_fixed=ZERO_NU_B, ν_u=1.75, ξ_u=2.25, ξ_W=0.00,
    ω̄=0.50, ω̄_star=0.25,
    κ=1.00, χ=0.0002, η=0.010,
    common_world_growth=false,
)

const AHP_CALIBRATION_SIGNATURE = (
    :pathwise_hi_exp_pi075,
    AHP_CALIBRATION.β, AHP_CALIBRATION.γ, AHP_CALIBRATION.π_persist,
    AHP_CALIBRATION.a_US, AHP_CALIBRATION.ϑ_US,
    AHP_CALIBRATION.a_W, AHP_CALIBRATION.H_W, AHP_CALIBRATION.L_W,
    AHP_CALIBRATION.A_X_US_u, AHP_CALIBRATION.A_L_US_u,
    AHP_CALIBRATION.ν_b_fixed, AHP_CALIBRATION.ν_u,
    AHP_CALIBRATION.ξ_u, AHP_CALIBRATION.ξ_W,
    AHP_CALIBRATION.ω̄, AHP_CALIBRATION.ω̄_star,
    AHP_CALIBRATION.κ, AHP_CALIBRATION.χ, AHP_CALIBRATION.η,
)

csv_factory_checks = Dict(
    "beta"=>AHP_CALIBRATION.β,
    "pi"=>AHP_CALIBRATION.π_persist,
    "a_W"=>AHP_CALIBRATION.a_W,
    "vartheta_US"=>AHP_CALIBRATION.ϑ_US,
    "nu_u"=>AHP_CALIBRATION.ν_u,
    "xi_u"=>AHP_CALIBRATION.ξ_u,
    "us_u_level"=>AHP_CALIBRATION.A_X_US_u / 10,
    "omega_bar"=>AHP_CALIBRATION.ω̄,
    "omega_bar_star"=>AHP_CALIBRATION.ω̄_star,
    "kappa"=>AHP_CALIBRATION.κ,
    "chi"=>AHP_CALIBRATION.χ,
    "eta"=>AHP_CALIBRATION.η,
)
AHP_SOURCE_NU_B_SEED = nb17_as_float(AHP_SELECTED_ROW["nu_b_seed"])
isapprox(AHP_SOURCE_NU_B_SEED, 0.10; atol=1e-12, rtol=1e-12) ||
    error("Unexpected source common-growth nu_b seed: $(AHP_SOURCE_NU_B_SEED)")

for (column, expected) in csv_factory_checks
    actual = nb17_as_float(AHP_SELECTED_ROW[column])
    isapprox(actual, expected; atol=1e-12, rtol=1e-12) ||
        error("Calibration provenance mismatch for $(column): CSV=$(actual), factory=$(expected)")
end

function parse_int_list(s::AbstractString)
    vals = Int[]
    for raw in split(s, ',')
        token = strip(raw)
        isempty(token) && continue
        push!(vals, parse(Int, token))
    end
    isempty(vals) && error("empty integer list")
    return sort(unique(vals))
end

int_list_literal(xs) = join(string.(xs), ",")

DEFAULT_BRANCH_ITER_SCHEDULE = RUN_MODE == :smoke ? [60, 200] : [500, 1000]
BRANCH_ITER_SCHEDULE = parse_int_list(get(
    ENV, "AHP_ZERO_NU_B_BRANCH_ITER_SCHEDULE", int_list_literal(DEFAULT_BRANCH_ITER_SCHEDULE)))
BRANCH_ITERS = first(BRANCH_ITER_SCHEDULE)
@assert all(>(0), BRANCH_ITER_SCHEDULE) "branch-iteration budgets must be positive"

# A fresh zero-exponent probe is safe through T=80 and residual-unsafe at T=81.
# The default cap brackets that transition without spending time on irrelevant
# larger horizons after the numerical boundary is already identified.
DEFAULT_SEARCH_CAP = RUN_MODE == :smoke ? 20 : 81
CONFIGURED_SEARCH_CAP = parse(Int, get(ENV, "AHP_ZERO_NU_B_SEARCH_CAP", string(DEFAULT_SEARCH_CAP)))
SEARCH_COARSE_STEP = parse(Int, get(ENV, "AHP_ZERO_NU_B_SEARCH_STEP", "5"))
REFINE_LOCAL_FRONTIER = lowercase(get(ENV, "AHP_ZERO_NU_B_REFINE_FRONTIER", "true")) in ("1", "true", "yes")
@assert CONFIGURED_SEARCH_CAP >= (RUN_MODE == :smoke ? 15 : 30) "search cap is below the calibrated anchor"
@assert SEARCH_COARSE_STEP > 0 "AHP_ZERO_NU_B_SEARCH_STEP must be positive"

HKT_E_RATIO_TOL = parse(Float64, get(ENV, "AHP_ZERO_NU_B_E_RATIO_TOL", "1e-4"))
HKT_QW_INCOME_RATIO_TOL = parse(Float64, get(
    ENV, "AHP_ZERO_NU_B_QW_INCOME_RATIO_TOL", string(HKT_E_RATIO_TOL)))
HKT_FUNDING_LIMIT_GAP_TOL = parse(Float64, get(ENV, "AHP_ZERO_NU_B_FUNDING_GAP_TOL", "1e-3"))
HKT_SUCCESSOR_PHI_LOG_GAP_TOL = parse(Float64, get(
    ENV, "AHP_ZERO_NU_B_SUCCESSOR_PHI_GAP_TOL", "1e-3"))
HKT_SCALAR_ROOT_TOL = parse(Float64, get(ENV, "AHP_ZERO_NU_B_SCALAR_ROOT_TOL", "1e-10"))
HKT_PREREQ_TAIL_WINDOW = parse(Int, get(
    ENV, "AHP_ZERO_NU_B_TAIL_WINDOW", RUN_MODE == :smoke ? "3" : "8"))
TAIL_REGRESSION_K_DEFAULT = RUN_MODE == :smoke ? 4 : 8
BOUND_EVALUATION_DATES = parse_int_list(get(
    ENV, "AHP_ZERO_NU_B_BOUND_DATES", "1,5,10,15"))
# The bubble-bound cutoff K is now measured back from the SELECTED horizon, not
# from the shortest solved one, so the guard below is what sets K. A large K is
# what makes the analytic HKT envelope usable at this calibration; the resulting
# short guard is checked by comparing the prefix product across every shorter
# hard-valid horizon that reaches K.
BOUND_ENDPOINT_GUARD = parse(Int, get(
    ENV, "AHP_ZERO_NU_B_BOUND_ENDPOINT_GUARD", RUN_MODE == :smoke ? "2" : "2"))
CONFIGURED_BOUND_CUTOFF = haskey(ENV, "AHP_ZERO_NU_B_BOUND_CUTOFF") ?
    parse(Int, ENV["AHP_ZERO_NU_B_BOUND_CUTOFF"]) : nothing
BOUND_PREFIX_STABILITY_TOL = parse(Float64, get(
    ENV, "AHP_ZERO_NU_B_BOUND_PREFIX_STABILITY_TOL", "1e-4"))

# zeta_lower and epsilon_q are cone choices, not primitives: the theory note
# calls zeta_lower = 1/2 only "the convenient cone choice", and this calibration
# violates both at 1/2 on its tail (zeta settles near 0.48). Derive them from the
# continuation window with a margin unless supplied; either way they stay
# conditional inputs and are labelled as such in every export.
BOUND_CONE_MARGIN = parse(Float64, get(ENV, "AHP_ZERO_NU_B_CONE_MARGIN", "0.05"))
CONFIGURED_ZETA_LOWER = haskey(ENV, "AHP_ZERO_NU_B_ZETA_LOWER") ?
    parse(Float64, ENV["AHP_ZERO_NU_B_ZETA_LOWER"]) : nothing
CONFIGURED_Q_PRICE_EPSILON = haskey(ENV, "AHP_ZERO_NU_B_Q_PRICE_EPSILON") ?
    parse(Float64, ENV["AHP_ZERO_NU_B_Q_PRICE_EPSILON"]) : nothing
CONFIGURED_LAMBDA_BOUND = haskey(ENV, "AHP_ZERO_NU_B_LAMBDA_RATIO_BOUND") ?
    parse(Float64, ENV["AHP_ZERO_NU_B_LAMBDA_RATIO_BOUND"]) : nothing
FUTURE_ENVELOPE_CERTIFIED = lowercase(get(
    ENV, "AHP_ZERO_NU_B_FUTURE_ENVELOPE_CERTIFIED", "false")) in ("1", "true", "yes")
@assert 0 < BOUND_CONE_MARGIN < 1
@assert CONFIGURED_ZETA_LOWER === nothing || 0 < CONFIGURED_ZETA_LOWER < 1
@assert CONFIGURED_Q_PRICE_EPSILON === nothing || 0 < CONFIGURED_Q_PRICE_EPSILON < 1
@assert BOUND_ENDPOINT_GUARD >= 2
@assert BOUND_PREFIX_STABILITY_TOL > 0
@assert CONFIGURED_LAMBDA_BOUND === nothing ||
        (isfinite(CONFIGURED_LAMBDA_BOUND) && CONFIGURED_LAMBDA_BOUND > 0)
@assert !FUTURE_ENVELOPE_CERTIFIED || CONFIGURED_LAMBDA_BOUND !== nothing     "certification requires a configured future-uniform lambda bound; nu_b=0 is fixed by the model"

function tail_combo_params(; T_max::Int, n_buffer::Int, branch_iters::Int=BRANCH_ITERS)
    c = AHP_CALIBRATION
    return ProductionParams(
        T_max=T_max, n_buffer=n_buffer,
        β=c.β, γ=c.γ, π_persist=c.π_persist,
        a_US=c.a_US, ϑ_US=c.ϑ_US,
        a_W=c.a_W, H_W=c.H_W, L_W=c.L_W,
        A_X_US_u=c.A_X_US_u, A_L_US_u=c.A_L_US_u,
        ν_b=c.ν_b_fixed, ν_u=c.ν_u, ξ_u=c.ξ_u, ξ_W=c.ξ_W,
        ω̄=c.ω̄, ω̄_star=c.ω̄_star,
        κ=c.κ, χ=c.χ, η=c.η,
        common_world_growth=c.common_world_growth,
        branch_iters=branch_iters, do_global_polish=false)
end

function default_horizon_candidates(cap::Int, step::Int)
    if RUN_MODE == :smoke
        return sort(unique(vcat([10, 15], collect(20:step:cap), [cap])))
    end
    return sort(unique(vcat(collect(30:step:cap), [cap])))
end

DEFAULT_HORIZON_T_CANDIDATES = default_horizon_candidates(CONFIGURED_SEARCH_CAP, SEARCH_COARSE_STEP)
CUSTOM_HORIZON_GRID = haskey(ENV, "AHP_ZERO_NU_B_T_CANDIDATES")
HORIZON_T_CANDIDATES = parse_int_list(get(
    ENV, "AHP_ZERO_NU_B_T_CANDIDATES", int_list_literal(DEFAULT_HORIZON_T_CANDIDATES)))
HORIZON_SEARCH_CAP = maximum(HORIZON_T_CANDIDATES)
VERIFY_SELECTED_COLD = lowercase(get(
    ENV, "AHP_ZERO_NU_B_VERIFY_SELECTED_COLD", "false")) in ("1", "true", "yes")
COLD_VERIFICATION_ATTEMPT_NOTE = get(
    ENV, "AHP_ZERO_NU_B_COLD_ATTEMPT_NOTE", "not_attempted")
AHP_REFERENCE_BRANCH_ITERS = parse(Int, get(
    ENV, "AHP_ZERO_NU_B_REFERENCE_BRANCH_ITERS", "60"))
VERIFY_AHP_BUFFERED_REFERENCE = lowercase(get(
    ENV, "AHP_ZERO_NU_B_VERIFY_BUFFERED_REFERENCE", "false")) in ("1", "true", "yes")
AHP_MATCH_TOL = RUN_MODE == :smoke ? 2e-5 : 1e-5
RUN_EXTENDED_TERMINAL_RULES = lowercase(get(
    ENV, "AHP_ZERO_NU_B_EXTENDED_TERMINAL_RULES", "false")) in ("1", "true", "yes")

HORIZON_TERMINAL_RULE = :local_loglinear
TERMINAL_RULES = RUN_EXTENDED_TERMINAL_RULES ?
    [:flat, :local_loglinear, :asymptotic_hkt, :tail_regression, :hkt_scalar_terminal_us] :
    [:local_loglinear, :hkt_scalar_terminal_us]

println("Notebook 15 winner:          ", AHP_SELECTED_LABEL)
println("AHP classification:          ", AHP_SELECTED_CLASSIFICATION)
println("Tracked t=15 rebased NFA:    ", AHP_EXPECTED_T15_REBASED_NFA)
println("Zero nu_b:                  ", ZERO_NU_B)
println("Source common-growth nu_b:   ", AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE)
println("Residual tolerance:          ", RESID_TOL)
println("Absorbing normalization tolerance: ", ABSORBING_INVARIANCE_TOL)
println("Branch-iteration schedule:   ", BRANCH_ITER_SCHEDULE)
println("No-buffer search grid:       T_report=", HORIZON_T_CANDIDATES)
println("HKT e_W/e_US threshold:      ", HKT_E_RATIO_TOL)
println("HKT Q_W/e_US threshold:      ", HKT_QW_INCOME_RATIO_TOL)
println("HKT zeta gap tolerance:      ", HKT_FUNDING_LIMIT_GAP_TOL)
println("Cold-verify selected branch: ", VERIFY_SELECTED_COLD)
println("Bubble-bound dates:          ", BOUND_EVALUATION_DATES)
println("Bubble-bound endpoint guard: ", BOUND_ENDPOINT_GUARD)
println("Configured bound cutoff:     ", CONFIGURED_BOUND_CUTOFF)
println("Bound cone margin:           ", BOUND_CONE_MARGIN)
println("Zeta lower source:           ",
        CONFIGURED_ZETA_LOWER === nothing ?
        "continuation minimum with margin (conditional)" : "environment")
println("q-price epsilon source:      ",
        CONFIGURED_Q_PRICE_EPSILON === nothing ?
        "continuation minimum with margin (conditional)" : "environment")
println("Reference branch iterations: ", AHP_REFERENCE_BRANCH_ITERS)
println("Cold buffered AHP replay:     ", VERIFY_AHP_BUFFERED_REFERENCE)
println("AHP path tolerance:           ", AHP_MATCH_TOL)
println("Uniform lambda bound source: ",
        CONFIGURED_LAMBDA_BOUND === nothing ? "finite-tail diagnostic (conditional)" : "environment")
println("Uniform nu_b bound:          fixed primitive p.nu_b = ", ZERO_NU_B)


## 2. Notebook-Local Terminal Closure Dispatcher

The terminal dispatcher matches Notebook 13, with one important interpretation in this notebook: because `n_buffer = 0`, `T_solve = T_report`, so the artificial successor is exactly `varphi_{US,T+1}` for the selected reported horizon.

In [ ]:
if !@isdefined TAIL_TERMINAL_RULE
    global TAIL_TERMINAL_RULE = Ref(:local_loglinear)
    global TAIL_TERMINAL_PARAMS = Ref{Any}(nothing)
    global TAIL_REGRESSION_K = Ref(TAIL_REGRESSION_K_DEFAULT)
else
    TAIL_TERMINAL_RULE[] = :local_loglinear
    TAIL_TERMINAL_PARAMS[] = nothing
    TAIL_REGRESSION_K[] = TAIL_REGRESSION_K_DEFAULT
end

function phi_clamp(x, p::ProductionParams)
    return clamp(Float64(x), p.φ_floor, 1 - p.φ_floor)
end

function phi_clamp(x)
    p = TAIL_TERMINAL_PARAMS[]
    return phi_clamp(x, p === nothing ? ProductionParams() : p)
end

function local_loglinear_phi(pol::Matrix{Float64}, i::Int, T::Int)
    if T >= 2
        a, b = pol[i, T - 1], pol[i, T]
        if isfinite(a) && isfinite(b) && a > 0 && b > 0
            return phi_clamp(b * b / a)
        end
    end
    return phi_clamp(pol[i, T])
end

function tail_regression_phi(pol::Matrix{Float64}, i::Int, T::Int; k::Int=8)
    lo = max(1, T - k + 1)
    idx = [t for t in lo:T if isfinite(pol[i, t]) && pol[i, t] > 0]
    length(idx) < 3 && return local_loglinear_phi(pol, i, T)

    x = Float64.(idx)
    y = log.([pol[i, t] for t in idx])
    xbar = mean(x); ybar = mean(y)
    sxx = sum(abs2, x .- xbar)
    sxx <= 0 && return local_loglinear_phi(pol, i, T)

    slope = sum((x .- xbar) .* (y .- ybar)) / sxx
    intercept = ybar - slope * xbar
    return phi_clamp(exp(intercept + slope * (T + 1)))
end

function asymptotic_hkt_phi_us(p::ProductionParams, phi_T::Real)
    phi_T = phi_clamp(phi_T, p)
    psi_US = (p.ξ_u - p.ν_u) * (p.ρ_US - 1)
    asymptotic_rate = G_N_US(p, phi_T)^(-psi_US / p.ρ_US)
    if isfinite(asymptotic_rate) && asymptotic_rate > 0
        return phi_clamp(phi_T * asymptotic_rate, p)
    end
    return phi_T
end

function asymptotic_hkt_phi_us(pol::Matrix{Float64}, T::Int)
    p = TAIL_TERMINAL_PARAMS[]
    p === nothing && return local_loglinear_phi(pol, 1, T)
    return asymptotic_hkt_phi_us(p, pol[1, T])
end

function hkt_scalar_equation_value(p::ProductionParams, N_US::Real, φ::Real)
    aH = p.a_US * p.H_US
    B = (p.A_X_US_u * p.H_US * N_US^p.ξ_u) /
        (p.A_L_US_u * p.L_US * N_US^p.ν_u)
    c = (p.β * (1 - p.α_US)) / (p.ϑ_US * p.α_US)
    return φ - 1 + p.β + c * B^(p.ρ_US - 1) * φ^p.ρ_US - 1 / aH
end

function hkt_scalar_phi_us(p::ProductionParams, N_US::Real)
    f(φ) = hkt_scalar_equation_value(p, N_US, φ)

    upper = 1 - p.φ_floor
    f(upper) <= 0 && return upper

    lo, hi = p.φ_floor, upper
    for _ in 1:240
        mid = 0.5 * (lo + hi)
        f(mid) > 0 ? (hi = mid) : (lo = mid)
    end
    return phi_clamp(0.5 * (lo + hi), p)
end

function hkt_scalar_successor_phi_us(pol::Matrix{Float64}, T::Int)
    p = TAIL_TERMINAL_PARAMS[]
    p === nothing && return local_loglinear_phi(pol, 1, T)

    # T is the solved horizon. The scalar HKT equation sets only the artificial
    # successor varphi_US,T+1; all dates 1:T are still solved in the full model.
    N_US_path = knowledge_path_US(p, vec(pol[1, 1:T]))
    N_US_succ = N_US_path[T + 1]
    return hkt_scalar_phi_us(p, N_US_succ)
end

function _extrapolate_terminal!(pol::Matrix{Float64}, T::Int)
    pol[:, T + 1] .= pol[:, T]
    rule = TAIL_TERMINAL_RULE[]

    if rule == :flat
        return pol
    elseif rule == :local_loglinear
        pol[1, T + 1] = local_loglinear_phi(pol, 1, T)
        pol[2, T + 1] = local_loglinear_phi(pol, 2, T)
    elseif rule == :asymptotic_hkt
        pol[1, T + 1] = asymptotic_hkt_phi_us(pol, T)
        pol[2, T + 1] = local_loglinear_phi(pol, 2, T)
    elseif rule == :tail_regression
        pol[1, T + 1] = tail_regression_phi(pol, 1, T; k=TAIL_REGRESSION_K[])
        pol[2, T + 1] = tail_regression_phi(pol, 2, T; k=TAIL_REGRESSION_K[])
    elseif rule == :hkt_scalar_terminal_us
        pol[1, T + 1] = hkt_scalar_successor_phi_us(pol, T)
        pol[2, T + 1] = local_loglinear_phi(pol, 2, T)
    else
        error("Unknown terminal rule: $rule")
    end
    return pol
end

function set_terminal_rule!(rule::Symbol, p::ProductionParams)
    @assert rule in TERMINAL_RULES "unknown terminal rule"
    TAIL_TERMINAL_RULE[] = rule
    TAIL_TERMINAL_PARAMS[] = p
    return rule
end

println("Installed notebook-local terminal closure dispatcher.")


## 3. Helpers

Each solve returns the equilibrium path, direct U.S. NFA accounting, absorbing-normalization diagnostics, and a compact certification row. It never calls `fundamental_value_path` or `nfa_decomposition`, because both would introduce the obsolete terminal fundamental-value anchor.


In [ ]:
path_vector(result, field::Symbol) = [getfield(s, field) for s in result.u_path]
path_vector_extended(result, field::Symbol) = [getfield(s, field) for s in result.u_path_extended]

function pretty(x)
    x === missing && return ""
    x isa Bool && return string(x)
    x isa Integer && return string(x)
    if x isa AbstractFloat
        if !isfinite(x)
            return string(x)
        elseif abs(x) >= 1000 || (abs(x) > 0 && abs(x) < 1e-4)
            return @sprintf("%.3e", x)
        else
            return @sprintf("%.6f", x)
        end
    end
    return string(x)
end

function html_escape(x)
    s = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    s = replace(s, "|" => "\\|")
    return replace(s, "_" => "\\_")
end

function markdown_table(rows, cols; maxrows=length(rows))
    isempty(rows) && return Base.HTML("<em>No rows.</em>")
    io = IOBuffer()
    println(io, "| ", join([html_escape(c) for c in cols], " | "), " |")
    println(io, "| ", join(fill("---", length(cols)), " | "), " |")
    for r in rows[1:min(maxrows, length(rows))]
        println(io, "| ",
                join([html_escape(pretty(getproperty(r, Symbol(c)))) for c in cols], " | "),
                " |")
    end
    return Base.HTML(repr(MIME"text/html"(), Markdown.parse(String(take!(io)))))
end

function csv_escape(x)
    x === missing && return ""
    s = string(x)
    if occursin(',', s) || occursin('"', s) || occursin('\n', s)
        return "\"" * replace(s, "\"" => "\"\"") * "\""
    end
    return s
end

function write_csv(filename, rows, cols)
    open(filename, "w") do io
        println(io, join(cols, ","))
        for r in rows
            println(io, join([csv_escape(getproperty(r, Symbol(c))) for c in cols], ","))
        end
    end
    return filename
end

function local_loglinear_next_value(x::AbstractVector{<:Real})
    length(x) >= 2 || return phi_clamp(x[end])
    a, b = x[end - 1], x[end]
    return isfinite(a) && isfinite(b) && a > 0 && b > 0 ?
        phi_clamp(b * b / a) : phi_clamp(b)
end

function tail_regression_next_value(x::AbstractVector{<:Real}; k::Int=8)
    T = length(x)
    lo = max(1, T - k + 1)
    idx = [t for t in lo:T if isfinite(x[t]) && x[t] > 0]
    length(idx) < 3 && return local_loglinear_next_value(x)
    xx = Float64.(idx)
    yy = log.([x[t] for t in idx])
    xbar, ybar = mean(xx), mean(yy)
    sxx = sum(abs2, xx .- xbar)
    sxx <= 0 && return local_loglinear_next_value(x)
    slope = sum((xx .- xbar) .* (yy .- ybar)) / sxx
    return phi_clamp(exp(ybar - slope * xbar + slope * (T + 1)))
end

function terminal_successor_N_US(result::ProductionSimulationResult)
    sT = result.u_path_extended[end]
    return G_N_US(result.params, sT.φ_US) * sT.N_US
end

function terminal_successor_N_W(result::ProductionSimulationResult)
    sT = result.u_path_extended[end]
    return G_N_W(result.params, sT.φ_W) * sT.N_W
end

function terminal_successor_phi_US(result::ProductionSimulationResult, rule::Symbol)
    vals = [s.φ_US for s in result.u_path_extended]
    if rule == :flat
        return phi_clamp(vals[end])
    elseif rule == :local_loglinear
        return local_loglinear_next_value(vals)
    elseif rule == :asymptotic_hkt
        return asymptotic_hkt_phi_us(result.params, vals[end])
    elseif rule == :tail_regression
        return tail_regression_next_value(vals; k=TAIL_REGRESSION_K[])
    elseif rule == :hkt_scalar_terminal_us
        return hkt_scalar_phi_us(result.params, terminal_successor_N_US(result))
    end
    error("Unknown terminal rule: $rule")
end

function terminal_successor_phi_W(result::ProductionSimulationResult, rule::Symbol)
    vals = [s.φ_W for s in result.u_path_extended]
    return rule == :flat ? phi_clamp(vals[end]) :
           rule == :tail_regression ? tail_regression_next_value(vals; k=TAIL_REGRESSION_K[]) :
           local_loglinear_next_value(vals)
end

finite_number(x) = x !== missing && x isa Real && isfinite(Float64(x))

function direct_nfa_decomposition(result::ProductionSimulationResult)
    p = result.params
    u = result.u_path
    A = Float64[s.A for s in u]
    Q_US = Float64[s.Q_US for s in u]
    Y_US = Float64[s.Y_US for s in u]
    NFA = A .- Q_US
    NFA_over_Y = NFA ./ Y_US
    rebased_NFA = (NFA .- first(NFA)) ./ Y_US

    portfolio_NFA = Float64[]
    for s in u
        n_W = (1 - s.ω) * (1 - s.θ) * s.A / s.q_W
        n_US_star = s.ω_star * (1 - s.θ_US_star) * s.A_star / s.q_US
        bond = s.θ * s.A
        push!(portfolio_NFA, s.q_W * n_W + bond - s.q_US * n_US_star)
    end

    return (;
        A, Q_US, Y_US, NFA, NFA_over_Y, rebased_NFA, portfolio_NFA,
        savings_identity_error=maximum(abs.(A .- p.β .* Float64[s.e_US for s in u])),
        portfolio_identity_error=maximum(abs.(NFA .- portfolio_NFA)))
end

function nb15_gate_accounting(result::ProductionSimulationResult, nfa)
    u = result.u_path
    T = length(u)
    T >= 3 || error("AHP accounting needs at least three periods.")
    q_US = Float64[s.q_US for s in u]
    q_W = Float64[s.q_W for s in u]
    Y_US = Float64[s.Y_US for s in u]
    n_W = Float64[(1 - s.ω) * (1 - s.θ) * s.A / s.q_W for s in u]
    n_US_star = Float64[
        s.ω_star * (1 - s.θ_US_star) * s.A_star / s.q_US for s in u]
    bond = Float64[s.θ * s.A for s in u]
    asset_position = q_W .* n_W
    liability_position = q_US .* n_US_star
    accounting_NFA = asset_position .+ bond .- liability_position
    VA_asset = zeros(T); VA_liability = zeros(T)
    CA_asset = zeros(T); CA_liability = zeros(T); CA_bond = zeros(T)
    for t in 2:T
        VA_asset[t] = n_W[t - 1] * (q_W[t] - q_W[t - 1])
        VA_liability[t] = -n_US_star[t - 1] * (q_US[t] - q_US[t - 1])
        CA_asset[t] = q_W[t] * (n_W[t] - n_W[t - 1])
        CA_liability[t] = -q_US[t] * (n_US_star[t] - n_US_star[t - 1])
        CA_bond[t] = bond[t] - bond[t - 1]
    end
    VA = VA_asset .+ VA_liability
    CA = CA_asset .+ CA_liability .+ CA_bond
    RES = zeros(T)
    for t in 2:T
        RES[t] = accounting_NFA[t] - accounting_NFA[t - 1] - CA[t] - VA[t]
    end
    return (; NFA=accounting_NFA, CA, VA, Y_US, n_US_star,
        liability_position, residual_max=maximum(abs.(RES[2:end])),
        nfa_identity_error=maximum(abs.(accounting_NFA .- nfa.NFA)))
end

function absorbing_normalization_summary(result::ProductionSimulationResult)
    p = result.params
    bgps = result.bgp_seq_extended
    reference = first(bgps)
    policy_fields = (:φ_US, :φ_W, :ω, :ω_star, :θ, :θ_US_star, :R_f, :R_f_W)
    aggregate_fields = (:Y_US, :Y_W, :e_US, :e_W, :Q_US, :Q_W,
                        :I_US, :I_W, :R_US, :R_W, :R_p, :R_A,
                        :R_p_star, :R_A_star, :G_N_US, :G_N_W, :Psi)
    policy_invariance_error = maximum(abs(getfield(b, f) - getfield(reference, f))
        for b in bgps for f in policy_fields)
    aggregate_invariance_error = maximum(abs(getfield(b, f) - getfield(reference, f))
        for b in bgps for f in aggregate_fields)
    per_variety_scaling_error = maximum(max(
        abs(b.N_US * b.q_US - reference.N_US * reference.q_US),
        abs(b.N_US * b.d_US - reference.N_US * reference.d_US),
        abs(b.N_W * b.q_W - reference.N_W * reference.q_W),
        abs(b.N_W * b.d_W - reference.N_W * reference.d_W),
    ) for b in bgps)
    return (;
        policy_invariance_error, aggregate_invariance_error,
        per_variety_scaling_error,
        all_extended_bgp_converged=all(b.converged for b in bgps),
        max_extended_bgp_residual=maximum(b.residual_norm for b in bgps),
    )
end

function nb15_hard_validity(result::ProductionSimulationResult, a)
    theta_US_star = path_vector(result, :θ_US_star)
    finite_accounting = all(isfinite, a.NFA) && all(isfinite, a.CA) &&
        all(isfinite, a.VA) && all(isfinite, a.Y_US)
    model_residuals = result.branch_converged &&
        isfinite(result.max_u_residual) && isfinite(result.max_bgp_residual) &&
        result.max_u_residual <= RESID_TOL && result.max_bgp_residual <= RESID_TOL
    psi_interior = result.diagnostics.psi_ok &&
        isfinite(result.diagnostics.psi_min) &&
        result.diagnostics.psi_min >= PSI_INTERIOR_TOL
    equity_interior = result.diagnostics.equity_weights_ok &&
        isfinite(result.diagnostics.equity_weight_min) &&
        result.diagnostics.equity_weight_min >= EQUITY_INTERIOR_TOL
    theta_interior = all(
        x -> isfinite(x) && THETA_INTERIOR_TOL <= x < 0.9, theta_US_star)
    phi_path = vcat(path_vector(result, :φ_US), path_vector(result, :φ_W))
    phi_lower_slack = minimum(phi_path) - result.params.φ_floor
    phi_upper_slack = (1 - result.params.φ_floor) - maximum(phi_path)
    phi_interior = min(phi_lower_slack, phi_upper_slack) >= PHI_INTERIOR_MARGIN
    positive_foreign_claims =
        all(x -> isfinite(x) && x > 0, a.n_US_star) &&
        all(x -> isfinite(x) && x > 0, a.liability_position)
    accounting_scale = max(1.0, maximum(abs.(a.NFA)),
        maximum(abs.(a.CA)), maximum(abs.(a.VA)))
    accounting_exact = isfinite(a.residual_max) &&
        a.residual_max <= RESID_TOL &&
        a.residual_max / accounting_scale <= ACCOUNTING_SCALED_TOL &&
        isfinite(a.nfa_identity_error) &&
        a.nfa_identity_error / accounting_scale <= ACCOUNTING_SCALED_TOL
    return_fields = (:R_A_u, :R_A_b, :R_A_star_u, :R_A_star_b, :R_f, :R_f_W)
    positive_returns = all(
        field -> all(x -> isfinite(x) && x > 0, path_vector(result, field)),
        return_fields)
    positive_output = all(x -> isfinite(x) && x > 0, a.Y_US)
    valid = model_residuals && psi_interior && equity_interior &&
        theta_interior && phi_interior && positive_foreign_claims && positive_returns &&
        accounting_exact && positive_output && finite_accounting
    return (; valid, model_residuals, psi_interior, equity_interior,
        theta_interior, phi_interior, phi_lower_slack, phi_upper_slack,
        positive_foreign_claims, positive_returns, accounting_exact,
        positive_output, finite_accounting, accounting_scale)
end

function nb15_zero_specialization_validity(result, a, absorbing)
    base = nb15_hard_validity(result, a)
    extended_bgp_valid = absorbing.all_extended_bgp_converged &&
        absorbing.max_extended_bgp_residual <= RESID_TOL
    normalization_valid = maximum((
        absorbing.policy_invariance_error,
        absorbing.aggregate_invariance_error,
        absorbing.per_variety_scaling_error,
    )) <= ABSORBING_INVARIANCE_TOL
    p = result.params
    primitive_exponent_ordering = p.ξ_u > p.ν_u > p.ν_b >= 0 &&
        p.ν_b == 0.0 && p.ξ_W == 0.0
    return merge(base, (; extended_bgp_valid, normalization_valid,
        primitive_exponent_ordering,
        valid=base.valid && extended_bgp_valid && normalization_valid &&
              primitive_exponent_ordering))
end

# Notebook 14's cache key omitted calibration values. Reset it unconditionally
# and retain the full calibration signature in every new key.
global RESULT_CACHE = Dict{Any,Any}()

function warm_start_fingerprint(warm_start_case)
    warm_start_case === nothing && return UInt(0)
    warm_start_case.status == :ok ||
        return hash((:unsafe_seed, warm_start_case.summary.T_report,
                     warm_start_case.summary.branch_iters))
    u = warm_start_case.result.u_path_extended
    isempty(u) && return UInt(0)
    return hash((warm_start_case.summary.T_report,
                 first(u).φ_US, first(u).φ_W, last(u).φ_US, last(u).φ_W,
                 warm_start_case.summary.max_u_residual))
end

function run_tail_case(T_report::Int, n_buffer::Int, rule::Symbol;
                       verbose::Bool=false,
                       branch_iters::Int=BRANCH_ITERS,
                       warm_start_case=nothing,
                       force::Bool=false)
    p = tail_combo_params(T_max=T_report, n_buffer=n_buffer, branch_iters=branch_iters)
    set_terminal_rule!(rule, p)
    seed_fingerprint = warm_start_fingerprint(warm_start_case)
    key = (AHP_CALIBRATION_SIGNATURE, T_report, n_buffer, rule,
           p.branch_iters, seed_fingerprint)
    !force && haskey(RESULT_CACHE, key) && return RESULT_CACHE[key]

    warm_ok = warm_start_case !== nothing && warm_start_case.status == :ok
    warm_start_T = warm_ok ? warm_start_case.summary.T_report : missing
    initial_u_path = warm_ok ? warm_start_case.result.u_path_extended : nothing
    @printf("Solving T=%d, buffer=%d, terminal=%s, branch_iters=%d, warm_T=%s\n",
            T_report, n_buffer, String(rule), branch_iters, pretty(warm_start_T))

    t0 = time()
    try
        result = run_production_simulation(p; verbose=verbose, initial_u_path=initial_u_path)
        nfa = direct_nfa_decomposition(result)
        absorbing = absorbing_normalization_summary(result)
        gate_accounting = nb15_gate_accounting(result, nfa)
        validity = nb15_zero_specialization_validity(result, gate_accounting, absorbing)
        residual_only = result.max_u_residual <= RESID_TOL &&
                        result.max_bgp_residual <= RESID_TOL
        hard_valid = validity.valid
        elapsed = time() - t0
        q = Float64.(path_vector(result, :q_US))
        successor_N_US = terminal_successor_N_US(result)
        successor_phi_US = terminal_successor_phi_US(result, rule)
        scalar_root_residual = rule == :hkt_scalar_terminal_us ?
            abs(hkt_scalar_equation_value(result.params, successor_N_US, successor_phi_US)) : missing
        rebased_t15 = length(nfa.rebased_NFA) >= 15 ? nfa.rebased_NFA[15] : missing
        summary = (;
            T_report=T_report,
            n_buffer=n_buffer,
            terminal_rule=String(rule),
            elapsed_sec=elapsed,
            status="ok",
            T_extended=length(result.u_path_extended),
            branch_iters=result.params.branch_iters,
            warm_start_T=warm_start_T,
            branch_converged=result.branch_converged,
            max_u_residual=result.max_u_residual,
            max_u_residual_t=argmax([s.residual_norm for s in result.u_path]),
            terminal_u_residual=result.u_path[end].residual_norm,
            max_bgp_residual=result.max_bgp_residual,
            residual_safe=residual_only,
            hard_valid=hard_valid,
            model_residuals=validity.model_residuals,
            psi_interior=validity.psi_interior,
            equity_interior=validity.equity_interior,
            theta_interior=validity.theta_interior,
            phi_interior=validity.phi_interior,
            phi_lower_slack=validity.phi_lower_slack,
            phi_upper_slack=validity.phi_upper_slack,
            positive_foreign_claims=validity.positive_foreign_claims,
            positive_returns=validity.positive_returns,
            accounting_exact=validity.accounting_exact,
            positive_output=validity.positive_output,
            finite_accounting=validity.finite_accounting,
            extended_bgp_valid=validity.extended_bgp_valid,
            normalization_valid=validity.normalization_valid,
            primitive_exponent_ordering=validity.primitive_exponent_ordering,
            accounting_residual_max=gate_accounting.residual_max,
            accounting_scale=validity.accounting_scale,
            nfa_identity_error=gate_accounting.nfa_identity_error,
            psi_ok=result.diagnostics.psi_ok,
            psi_min=result.diagnostics.psi_min,
            equity_weights_ok=result.diagnostics.equity_weights_ok,
            equity_weight_min=result.diagnostics.equity_weight_min,
            nu_b_reference=result.params.ν_b,
            absorbing_policy_invariance_error=absorbing.policy_invariance_error,
            absorbing_aggregate_invariance_error=absorbing.aggregate_invariance_error,
            absorbing_per_variety_scaling_error=absorbing.per_variety_scaling_error,
            all_extended_bgp_converged=absorbing.all_extended_bgp_converged,
            max_extended_bgp_residual=absorbing.max_extended_bgp_residual,
            terminal_successor_scope="T_report_plus_1_only",
            terminal_successor_N_US=successor_N_US,
            terminal_successor_phi_US=successor_phi_US,
            terminal_successor_phi_W=terminal_successor_phi_W(result, rule),
            hkt_scalar_root_residual=scalar_root_residual,
            phi_US_report_end=result.u_path[end].φ_US,
            q_US_report_end=result.u_path[end].q_US,
            q_growth=q[end] / q[1],
            nfa_final=nfa.NFA[end],
            nfa_min=minimum(nfa.NFA),
            rebased_nfa_t15=rebased_t15,
            nfa_savings_identity_error=nfa.savings_identity_error,
            nfa_portfolio_identity_error=nfa.portfolio_identity_error,
            cond_1b_final=result.diagnostics.cond_1b[end],
            error="")
        out = (; status=:ok, result, nfa, absorbing, summary)
        RESULT_CACHE[key] = out
        return out
    catch err
        elapsed = time() - t0
        summary = (;
            T_report=T_report, n_buffer=n_buffer, terminal_rule=String(rule),
            elapsed_sec=elapsed, status="error", T_extended=missing,
            branch_iters=p.branch_iters, warm_start_T=warm_start_T,
            branch_converged=missing, max_u_residual=missing,
            max_u_residual_t=missing, terminal_u_residual=missing,
            max_bgp_residual=missing, residual_safe=false, hard_valid=false,
            model_residuals=false, psi_interior=false, equity_interior=false,
            theta_interior=false, phi_interior=false,
            phi_lower_slack=missing, phi_upper_slack=missing,
            positive_foreign_claims=false, positive_returns=false,
            accounting_exact=false, positive_output=false, finite_accounting=false,
            extended_bgp_valid=false, normalization_valid=false,
            primitive_exponent_ordering=false, accounting_residual_max=missing,
            accounting_scale=missing, nfa_identity_error=missing,
            psi_ok=missing, psi_min=missing, equity_weights_ok=missing,
            equity_weight_min=missing, nu_b_reference=missing,
            absorbing_policy_invariance_error=missing,
            absorbing_aggregate_invariance_error=missing,
            absorbing_per_variety_scaling_error=missing,
            all_extended_bgp_converged=missing, max_extended_bgp_residual=missing,
            terminal_successor_scope="T_report_plus_1_only",
            terminal_successor_N_US=missing, terminal_successor_phi_US=missing,
            terminal_successor_phi_W=missing, hkt_scalar_root_residual=missing,
            phi_US_report_end=missing, q_US_report_end=missing, q_growth=missing,
            nfa_final=missing, nfa_min=missing, rebased_nfa_t15=missing,
            nfa_savings_identity_error=missing, nfa_portfolio_identity_error=missing,
            cond_1b_final=missing, error=sprint(showerror, err))
        out = (; status=:error, result=nothing, nfa=nothing, absorbing=nothing, summary)
        RESULT_CACHE[key] = out
        @warn "Solve failed" T_report n_buffer rule branch_iters err
        return out
    end
end

function residual_safe(case)
    return case.status == :ok && case.summary.hard_valid === true
end

function residual_score(case)
    case.status == :ok || return Inf
    u = finite_number(case.summary.max_u_residual) ? Float64(case.summary.max_u_residual) : Inf
    b = finite_number(case.summary.max_bgp_residual) ? Float64(case.summary.max_bgp_residual) : Inf
    return max(u, b)
end

function run_residual_safe_case(T_report::Int, n_buffer::Int, rule::Symbol;
                                warm_start_case=nothing, verbose::Bool=false)
    attempts = Any[]
    seed = warm_start_case
    for branch_iters in BRANCH_ITER_SCHEDULE
        c = run_tail_case(T_report, n_buffer, rule;
                          verbose=verbose, branch_iters=branch_iters,
                          warm_start_case=seed)
        push!(attempts, c)
        residual_safe(c) && return (; case=c, attempts)
        c.status == :ok && (seed = c)
    end
    return (; case=attempts[argmin(residual_score.(attempts))], attempts)
end

summary_cols = [
    "T_report", "n_buffer", "terminal_rule", "elapsed_sec", "status",
    "T_extended", "branch_iters", "warm_start_T", "branch_converged",
    "max_u_residual", "max_u_residual_t", "terminal_u_residual",
    "max_bgp_residual", "residual_safe", "hard_valid",
    "model_residuals", "psi_interior", "equity_interior",
    "theta_interior", "phi_interior", "phi_lower_slack", "phi_upper_slack",
    "positive_foreign_claims", "positive_returns",
    "accounting_exact", "positive_output", "finite_accounting",
    "extended_bgp_valid", "normalization_valid",
    "primitive_exponent_ordering", "accounting_residual_max",
    "accounting_scale", "nfa_identity_error",
    "psi_ok", "psi_min",
    "equity_weights_ok", "equity_weight_min", "nu_b_reference",
    "absorbing_policy_invariance_error", "absorbing_aggregate_invariance_error",
    "absorbing_per_variety_scaling_error", "all_extended_bgp_converged",
    "max_extended_bgp_residual",
    "terminal_successor_scope", "terminal_successor_N_US",
    "terminal_successor_phi_US", "terminal_successor_phi_W",
    "hkt_scalar_root_residual", "phi_US_report_end", "q_US_report_end",
    "q_growth", "nfa_final", "nfa_min", "rebased_nfa_t15",
    "nfa_savings_identity_error", "nfa_portfolio_identity_error",
    "cond_1b_final", "error"]

println("Installed direct-NFA, zero-normalization, solve, cache, table, and successor helpers.")


## 4. Long Hard-Valid No-Buffer Search and AHP NFA Reproduction

The search starts at period 30, the horizon on which Notebook 15 cold-ranks its finalists, rather than assuming Notebook 14's obsolete \(45{:}60\) baseline remains appropriate. After selecting the longest hard-valid path, the notebook recomputes signed U.S. NFA directly and compares its first 15 periods with Notebook 15's tracked path.


In [ ]:
function execute_horizon_search()
    horizon_cases = Any[]
    horizon_attempts = Any[]
    horizon_stage = Dict{Int,String}()
    warm_case = nothing

    function write_horizon_progress!(cases, attempts, stage)
        chosen_rows = [merge(c.summary, (;
            search_stage=get(stage, c.summary.T_report, "coarse"),
            selected_for_downstream=false,
            search_right_censored=false)) for c in sort(cases; by=c -> c.summary.T_report)]
        attempt_rows = [merge(c.summary, (;
            search_stage=get(stage, c.summary.T_report, "coarse"))) for c in attempts]
        chosen_cols = vcat(summary_cols, ["search_stage", "selected_for_downstream", "search_right_censored"])
        attempt_cols = vcat(summary_cols, ["search_stage"])
        write_csv(joinpath(OUTDIR, "residual_safe_horizon_progress.csv"), chosen_rows, chosen_cols)
        write_csv(joinpath(OUTDIR, "residual_safe_horizon_attempts.csv"), attempt_rows, attempt_cols)
        return nothing
    end

    for T_report in HORIZON_T_CANDIDATES
        outcome = run_residual_safe_case(T_report, NO_BUFFER, HORIZON_TERMINAL_RULE;
                                         warm_start_case=warm_case)
        push!(horizon_cases, outcome.case)
        append!(horizon_attempts, outcome.attempts)
        horizon_stage[T_report] = "coarse_grid"
        residual_safe(outcome.case) && (warm_case = outcome.case)
        write_horizon_progress!(horizon_cases, horizon_attempts, horizon_stage)
    end

    safe_horizon_cases = filter(residual_safe, horizon_cases)
    if !isempty(safe_horizon_cases) && REFINE_LOCAL_FRONTIER
        coarse_safe_T = maximum(c.summary.T_report for c in safe_horizon_cases)
        cap_case = only(filter(c -> c.summary.T_report == HORIZON_SEARCH_CAP, horizon_cases))
        if coarse_safe_T < HORIZON_SEARCH_CAP && !residual_safe(cap_case)
            solved_T = Set(c.summary.T_report for c in horizon_cases)
            warm_case = only(filter(c -> c.summary.T_report == coarse_safe_T, safe_horizon_cases))
            for T_report in (coarse_safe_T + 1):HORIZON_SEARCH_CAP
                T_report in solved_T && continue
                outcome = run_residual_safe_case(T_report, NO_BUFFER, HORIZON_TERMINAL_RULE;
                                                 warm_start_case=warm_case)
                push!(horizon_cases, outcome.case)
                append!(horizon_attempts, outcome.attempts)
                horizon_stage[T_report] = "exhaustive_upper_frontier"
                residual_safe(outcome.case) && (warm_case = outcome.case)
                write_horizon_progress!(horizon_cases, horizon_attempts, horizon_stage)
            end
        end
    end

    sort!(horizon_cases; by=c -> c.summary.T_report)
    safe_horizon_cases = filter(residual_safe, horizon_cases)

    isempty(safe_horizon_cases) && begin
        write_horizon_progress!(horizon_cases, horizon_attempts, horizon_stage)
        display(markdown_table([c.summary for c in horizon_cases],
            ["T_report", "branch_iters", "warm_start_T", "status", "max_u_residual",
             "max_bgp_residual", "residual_safe", "error"];
            maxrows=length(horizon_cases)))
        error("No residual-safe no-buffer horizon found in the explored grid. Diagnostics were saved before this error.")
    end

    SELECTED_T_REPORT = maximum(c.summary.T_report for c in safe_horizon_cases)
    selected_horizon_case = only(filter(c -> c.summary.T_report == SELECTED_T_REPORT, safe_horizon_cases))
    SEARCH_RIGHT_CENSORED = SELECTED_T_REPORT == HORIZON_SEARCH_CAP && residual_safe(selected_horizon_case)
    cap_case = only(filter(c -> c.summary.T_report == HORIZON_SEARCH_CAP, horizon_cases))
    SEARCH_OUTCOME = SEARCH_RIGHT_CENSORED ? "safe_at_cap_right_censored" :
        (cap_case.status == :error ? "cap_solve_error" : "cap_residual_unsafe_within_iteration_budget")

    cold_verification = if VERIFY_SELECTED_COLD
        cold = run_tail_case(SELECTED_T_REPORT, NO_BUFFER, HORIZON_TERMINAL_RULE;
                             branch_iters=selected_horizon_case.summary.branch_iters,
                             warm_start_case=nothing, force=true)
        if residual_safe(cold)
            Tcmp = min(length(cold.result.u_path), length(selected_horizon_case.result.u_path))
            phi_gap = maximum(abs(log(cold.result.u_path[t].φ_US) -
                                  log(selected_horizon_case.result.u_path[t].φ_US)) for t in 1:Tcmp)
            q_gap = maximum(abs(log(cold.result.u_path[t].q_US) -
                                log(selected_horizon_case.result.u_path[t].q_US)) for t in 1:Tcmp)
            (; cold_verification_status="residual_safe", cold_max_u_residual=cold.summary.max_u_residual,
               cold_max_bgp_residual=cold.summary.max_bgp_residual,
               cold_vs_selected_max_log_phi_US_gap=phi_gap,
               cold_vs_selected_max_log_q_US_gap=q_gap)
        else
            (; cold_verification_status="failed_or_residual_unsafe",
               cold_max_u_residual=cold.summary.max_u_residual,
               cold_max_bgp_residual=cold.summary.max_bgp_residual,
               cold_vs_selected_max_log_phi_US_gap=missing,
               cold_vs_selected_max_log_q_US_gap=missing)
        end
    else
        (; cold_verification_status="not_requested", cold_max_u_residual=missing,
           cold_max_bgp_residual=missing, cold_vs_selected_max_log_phi_US_gap=missing,
           cold_vs_selected_max_log_q_US_gap=missing)
    end

    if VERIFY_SELECTED_COLD
        @assert cold_verification.cold_verification_status == "residual_safe" "selected cold solve is not hard-valid"
        @assert cold_verification.cold_vs_selected_max_log_phi_US_gap <= COLD_PATH_TOL
        @assert cold_verification.cold_vs_selected_max_log_q_US_gap <= COLD_PATH_TOL
    end

    horizon_search_rows = [merge(c.summary, (;
        search_stage=get(horizon_stage, c.summary.T_report, "coarse_grid"),
        selected_for_downstream=c.summary.T_report == SELECTED_T_REPORT,
        search_right_censored=c.summary.T_report == SELECTED_T_REPORT && SEARCH_RIGHT_CENSORED,
        search_outcome=SEARCH_OUTCOME))
        for c in horizon_cases]
    horizon_search_cols = vcat(summary_cols,
        ["search_stage", "selected_for_downstream", "search_right_censored", "search_outcome"])
    write_csv(joinpath(OUTDIR, "residual_safe_horizon_search.csv"), horizon_search_rows, horizon_search_cols)

    selected_horizon_row = merge(selected_horizon_case.summary, (;
        search_stage=get(horizon_stage, SELECTED_T_REPORT, "coarse_grid"),
        selected_for_downstream=true,
        search_right_censored=SEARCH_RIGHT_CENSORED,
        search_outcome=SEARCH_OUTCOME))
    write_csv(joinpath(OUTDIR, "selected_horizon_summary.csv"),
              [selected_horizon_row], horizon_search_cols)
    cold_cols = ["cold_verification_status", "cold_max_u_residual", "cold_max_bgp_residual",
                 "cold_vs_selected_max_log_phi_US_gap", "cold_vs_selected_max_log_q_US_gap"]
    write_csv(joinpath(OUTDIR, "selected_horizon_cold_verification.csv"),
              [cold_verification], cold_cols)

    display(markdown_table(horizon_search_rows,
        ["T_report", "search_stage", "branch_iters", "warm_start_T", "status",
         "max_u_residual", "max_u_residual_t", "terminal_u_residual",
         "max_bgp_residual", "residual_safe", "selected_for_downstream",
         "search_right_censored"];
        maxrows=length(horizon_search_rows)))

    println("Largest residual-safe no-buffer horizon in the explored grid = ", SELECTED_T_REPORT)
    println("Search outcome: ", SEARCH_OUTCOME)
    if SEARCH_RIGHT_CENSORED
        if CUSTOM_HORIZON_GRID
            println("RIGHT-CENSORED: extend AHP_ZERO_NU_B_T_CANDIDATES to search farther than T=", HORIZON_SEARCH_CAP, ".")
        else
            println("RIGHT-CENSORED: raise AHP_ZERO_NU_B_SEARCH_CAP to search farther than T=", HORIZON_SEARCH_CAP, ".")
        end
    else
        println("The cap was not certified safe; this is a within-budget numerical classification, not an economic horizon bound.")
    end
    println("Selected-branch cold verification: ", cold_verification.cold_verification_status)
    return (; horizon_cases, horizon_attempts, horizon_stage, safe_horizon_cases,
              SELECTED_T_REPORT, selected_horizon_case, SEARCH_RIGHT_CENSORED, SEARCH_OUTCOME,
              horizon_search_rows, horizon_search_cols, cold_verification)
end

horizon_search_state = execute_horizon_search()
horizon_cases = horizon_search_state.horizon_cases
horizon_attempts = horizon_search_state.horizon_attempts
horizon_stage = horizon_search_state.horizon_stage
safe_horizon_cases = horizon_search_state.safe_horizon_cases
SELECTED_T_REPORT = horizon_search_state.SELECTED_T_REPORT
selected_horizon_case = horizon_search_state.selected_horizon_case
SEARCH_RIGHT_CENSORED = horizon_search_state.SEARCH_RIGHT_CENSORED
SEARCH_OUTCOME = horizon_search_state.SEARCH_OUTCOME
selected_horizon_cold_verification = horizon_search_state.cold_verification
horizon_search_rows = horizon_search_state.horizon_search_rows
horizon_search_cols = horizon_search_state.horizon_search_cols

# Reproduce Notebook 15's signed first-window NFA object without invoking
# any fundamental-value recursion. The first solved T=30 no-buffer path is
# the default cold anchor; an exact buffer-10 replay is available on request.
tracked_data, tracked_names = nb17_read_csv_any(NB15_TRACKED_PATH_CSV)
tracked_t_col = nb17_column_index(tracked_names, "t")
tracked_nfa_col = nb17_column_index(tracked_names, "rebased_NFA_change_current_Y")
tracked_t = Int[nb17_as_int(tracked_data[i, tracked_t_col]) for i in axes(tracked_data, 1)]
tracked_rebased_nfa = Float64[nb17_as_float(tracked_data[i, tracked_nfa_col])
                              for i in axes(tracked_data, 1)]
tracked_first_window = Dict(tracked_t[i] => tracked_rebased_nfa[i]
                            for i in eachindex(tracked_t) if tracked_t[i] <= 15)
AHP_MATCH_DATES = collect(1:15)
sort(collect(keys(tracked_first_window))) == AHP_MATCH_DATES ||
    error("Notebook 15 tracked NFA file must contain every date 1:15 exactly once")

reference_candidates = filter(
    c -> c.summary.T_report >= 30, sort(safe_horizon_cases; by=c -> c.summary.T_report))
ahp_reference_kind = VERIFY_AHP_BUFFERED_REFERENCE ?
    "cold_T30_buffer10_replay" :
    (isempty(reference_candidates) ? "smoke_selected_path" : "cold_no_buffer_grid_anchor")
ahp_reference_case = if VERIFY_AHP_BUFFERED_REFERENCE
    run_tail_case(30, 10, HORIZON_TERMINAL_RULE;
        branch_iters=AHP_REFERENCE_BRANCH_ITERS, warm_start_case=nothing, force=true)
elseif isempty(reference_candidates)
    selected_horizon_case
else
    first(reference_candidates)
end
residual_safe(ahp_reference_case) ||
    error("AHP calibration reference path is not hard-valid")
isapprox(ahp_reference_case.result.params.ν_b, ZERO_NU_B;
         atol=1e-8, rtol=1e-8) ||
    error("Zero-nu_b solve did not preserve nu_b=0.0")
length(ahp_reference_case.nfa.rebased_NFA) >= 15 ||
    error("Dedicated AHP reference solve does not cover periods 1:15")
length(selected_horizon_case.nfa.rebased_NFA) >= 15 ||
    error("Selected no-buffer path does not cover periods 1:15")

reference_rebased_nfa = ahp_reference_case.nfa.rebased_NFA
selected_rebased_nfa = selected_horizon_case.nfa.rebased_NFA
AHP_REFERENCE_MAX_FIRST_WINDOW_PATH_ERROR = maximum(
    abs(reference_rebased_nfa[t] - tracked_first_window[t]) for t in AHP_MATCH_DATES)
AHP_SELECTED_MAX_FIRST_WINDOW_PATH_ERROR = maximum(
    abs(selected_rebased_nfa[t] - tracked_first_window[t]) for t in AHP_MATCH_DATES)
AHP_CURRENT_T15_REBASED_NFA = selected_rebased_nfa[15]
AHP_T15_ERROR = AHP_CURRENT_T15_REBASED_NFA - AHP_EXPECTED_T15_REBASED_NFA
AHP_SOURCE_PATH_MATCHED =
    AHP_REFERENCE_MAX_FIRST_WINDOW_PATH_ERROR <= AHP_MATCH_TOL &&
    AHP_SELECTED_MAX_FIRST_WINDOW_PATH_ERROR <= AHP_MATCH_TOL

ahp_path_rows = [(;
    t=t, model_t=t - 1,
    tracked_rebased_nfa=tracked_first_window[t],
    reference_rebased_nfa=reference_rebased_nfa[t],
    selected_no_buffer_rebased_nfa=selected_rebased_nfa[t],
    reference_error=reference_rebased_nfa[t] - tracked_first_window[t],
    selected_error=selected_rebased_nfa[t] - tracked_first_window[t])
    for t in AHP_MATCH_DATES]
ahp_path_cols = [
    "t", "model_t", "tracked_rebased_nfa", "reference_rebased_nfa",
    "selected_no_buffer_rebased_nfa", "reference_error", "selected_error"]
write_csv(joinpath(OUTDIR, "ahp_nfa_zero_nu_b_comparison_path.csv"),
          ahp_path_rows, ahp_path_cols)

ahp_validation_rows = [(;
    selected_label=AHP_SELECTED_LABEL,
    classification=AHP_SELECTED_CLASSIFICATION,
    evidence_window_end=15,
    reference_kind=ahp_reference_kind,
    reference_T_report=ahp_reference_case.summary.T_report,
    reference_n_buffer=ahp_reference_case.summary.n_buffer,
    reference_branch_iters=ahp_reference_case.summary.branch_iters,
    match_tolerance=AHP_MATCH_TOL,
    selected_T_report=SELECTED_T_REPORT,
    zero_nu_b=ZERO_NU_B,
    source_common_growth_nu_b_reference=AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE,
    current_zero_nu_b=ahp_reference_case.result.params.ν_b,
    tracked_t15_rebased_nfa=AHP_EXPECTED_T15_REBASED_NFA,
    current_t15_rebased_nfa=AHP_CURRENT_T15_REBASED_NFA,
    t15_error=AHP_T15_ERROR,
    reference_max_first_window_path_error=AHP_REFERENCE_MAX_FIRST_WINDOW_PATH_ERROR,
    selected_max_first_window_path_error=AHP_SELECTED_MAX_FIRST_WINDOW_PATH_ERROR,
    matches_source_within_tolerance=AHP_SOURCE_PATH_MATCHED,
    reference_hard_valid=residual_safe(ahp_reference_case),
    selected_hard_valid=residual_safe(selected_horizon_case),
    reference_nfa_savings_identity_error=ahp_reference_case.nfa.savings_identity_error,
    reference_nfa_portfolio_identity_error=ahp_reference_case.nfa.portfolio_identity_error,
    selected_nfa_savings_identity_error=selected_horizon_case.nfa.savings_identity_error,
    selected_nfa_portfolio_identity_error=selected_horizon_case.nfa.portfolio_identity_error)]
ahp_validation_cols = String.(propertynames(first(ahp_validation_rows)))
write_csv(joinpath(OUTDIR, "ahp_nfa_zero_nu_b_comparison_summary.csv"),
          ahp_validation_rows, ahp_validation_cols)
display(markdown_table(ahp_validation_rows, ahp_validation_cols;
                       maxrows=length(ahp_validation_rows)))

p_ref = ahp_reference_case.result.params
calibration_manifest_rows = [
    (; parameter="selected_label", configured=AHP_SELECTED_LABEL, resolved=AHP_SELECTED_LABEL, source="Notebook 15 candidate_summary.csv"),
    (; parameter="beta", configured=AHP_CALIBRATION.β, resolved=p_ref.β, source="Notebook 16 factory"),
    (; parameter="gamma", configured=AHP_CALIBRATION.γ, resolved=p_ref.γ, source="Notebook 16 factory"),
    (; parameter="pi_persist", configured=AHP_CALIBRATION.π_persist, resolved=p_ref.π_persist, source="Notebook 16 factory"),
    (; parameter="a_US", configured=AHP_CALIBRATION.a_US, resolved=p_ref.a_US, source="Notebook 16 factory"),
    (; parameter="vartheta_US", configured=AHP_CALIBRATION.ϑ_US, resolved=p_ref.ϑ_US, source="Notebook 16 factory"),
    (; parameter="a_W", configured=AHP_CALIBRATION.a_W, resolved=p_ref.a_W, source="Notebook 16 factory"),
    (; parameter="H_W", configured=AHP_CALIBRATION.H_W, resolved=p_ref.H_W, source="Notebook 16 factory"),
    (; parameter="L_W", configured=AHP_CALIBRATION.L_W, resolved=p_ref.L_W, source="Notebook 16 factory"),
    (; parameter="A_X_US_u", configured=AHP_CALIBRATION.A_X_US_u, resolved=p_ref.A_X_US_u, source="Notebook 16 factory"),
    (; parameter="A_L_US_u", configured=AHP_CALIBRATION.A_L_US_u, resolved=p_ref.A_L_US_u, source="Notebook 16 factory"),
    (; parameter="nu_b_fixed_primitive", configured=AHP_CALIBRATION.ν_b_fixed, resolved=AHP_CALIBRATION.ν_b_fixed, source="fixed primitive user instruction"),
    (; parameter="source_common_growth_nu_b_reference", configured=AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE, resolved=p_ref.ν_b, source="source Notebook 15 benchmark versus fixed solver"),
    (; parameter="nu_u", configured=AHP_CALIBRATION.ν_u, resolved=p_ref.ν_u, source="Notebook 16 factory"),
    (; parameter="xi_u", configured=AHP_CALIBRATION.ξ_u, resolved=p_ref.ξ_u, source="Notebook 16 factory"),
    (; parameter="xi_W", configured=AHP_CALIBRATION.ξ_W, resolved=p_ref.ξ_W, source="Notebook 16 factory"),
    (; parameter="omega_bar", configured=AHP_CALIBRATION.ω̄, resolved=p_ref.ω̄, source="Notebook 16 factory"),
    (; parameter="omega_bar_star", configured=AHP_CALIBRATION.ω̄_star, resolved=p_ref.ω̄_star, source="Notebook 16 factory"),
    (; parameter="kappa", configured=AHP_CALIBRATION.κ, resolved=p_ref.κ, source="Notebook 16 factory"),
    (; parameter="chi", configured=AHP_CALIBRATION.χ, resolved=p_ref.χ, source="Notebook 16 factory"),
    (; parameter="eta", configured=AHP_CALIBRATION.η, resolved=p_ref.η, source="Notebook 16 factory"),
    (; parameter="common_world_growth", configured=AHP_CALIBRATION.common_world_growth, resolved=p_ref.common_world_growth, source="inert compatibility metadata; omitted from the zero-exponent solver"),
]
write_csv(joinpath(OUTDIR, "calibration_manifest.csv"), calibration_manifest_rows,
          ["parameter", "configured", "resolved", "source"])

run_manifest_rows = [
    (; field="run_started_utc", value=RUN_STARTED_UTC),
    (; field="run_mode", value=String(RUN_MODE)),
    (; field="project_dir", value=PROJECT_DIR),
    (; field="model_file", value=MODEL_FILE),
    (; field="model_file_mtime_unix", value=stat(MODEL_FILE).mtime),
    (; field="horizon_candidates", value=int_list_literal(HORIZON_T_CANDIDATES)),
    (; field="custom_horizon_grid", value=CUSTOM_HORIZON_GRID),
    (; field="horizon_search_cap", value=HORIZON_SEARCH_CAP),
    (; field="search_right_censored", value=SEARCH_RIGHT_CENSORED),
    (; field="search_outcome", value=SEARCH_OUTCOME),
    (; field="branch_iter_schedule", value=int_list_literal(BRANCH_ITER_SCHEDULE)),
    (; field="verify_selected_cold", value=VERIFY_SELECTED_COLD),
    (; field="reference_branch_iters", value=AHP_REFERENCE_BRANCH_ITERS),
    (; field="verify_buffered_reference", value=VERIFY_AHP_BUFFERED_REFERENCE),
    (; field="ahp_match_tolerance", value=AHP_MATCH_TOL),
    (; field="residual_tolerance", value=RESID_TOL),
    (; field="hard_validity_tolerance", value=CERT_TOL),
    (; field="bound_endpoint_guard", value=BOUND_ENDPOINT_GUARD),
    (; field="configured_bound_cutoff", value=CONFIGURED_BOUND_CUTOFF),
    (; field="future_envelope_certified", value=FUTURE_ENVELOPE_CERTIFIED),
]
write_csv(joinpath(OUTDIR, "run_manifest.csv"), run_manifest_rows, ["field", "value"])

p_ahp_nfa = plot(AHP_MATCH_DATES,
    [tracked_first_window[t] for t in AHP_MATCH_DATES],
    lw=2.6, color=:black, ls=:dash, marker=:circle,
    label="source common-growth AHP path",
    title="Zero nu_b=0 U.S. NFA path versus source AHP benchmark",
    xlabel="notebook period t (model date t-1)",
    ylabel="(NFA_t-NFA_1)/Y_US,t")
plot!(p_ahp_nfa, AHP_MATCH_DATES, reference_rebased_nfa[AHP_MATCH_DATES],
      lw=2.0, color=:darkorange, label="zero-nu_b anchor: " * ahp_reference_kind)
plot!(p_ahp_nfa, AHP_MATCH_DATES, selected_rebased_nfa[AHP_MATCH_DATES],
      lw=2.2, color=:navy, label="selected no-buffer tail path")
hline!(p_ahp_nfa, [0.0], color=:gray45, ls=:dot, lw=1.0, label="zero")
savefig(p_ahp_nfa, joinpath(OUTDIR, "ahp_nfa_zero_nu_b_comparison.png"))
display(p_ahp_nfa)

println("Zero-nu_b path matches the source common-growth AHP path within tolerance: ",
        AHP_SOURCE_PATH_MATCHED)
println("This comparison is diagnostic and is not a model-validity gate.")


## 4.1 Solve-Once Absorbing Normalization Check

The following check targets the mechanism created by $\nu_b=\xi_W=0$. It
verifies that all switch-state records reuse one policy/aggregate solution and
that only per-variety prices and dividends scale with inherited knowledge.
This diagnostic is evaluated after solving and adds no equilibrium equation.


In [ ]:
absorbing_case = selected_horizon_case
absorbing_bgps = absorbing_case.result.bgp_seq_extended
absorbing_ref = first(absorbing_bgps)
absorbing_rows = [(;
    switch_index=j - 1,
    N_US=b.N_US, N_W=b.N_W,
    phi_US=b.φ_US, phi_W=b.φ_W,
    q_US=b.q_US, d_US=b.d_US, q_W=b.q_W, d_W=b.d_W,
    Nq_US=b.N_US * b.q_US, Nd_US=b.N_US * b.d_US,
    Nq_W=b.N_W * b.q_W, Nd_W=b.N_W * b.d_W,
    Q_US=b.Q_US, Q_W=b.Q_W,
) for (j, b) in enumerate(absorbing_bgps)]

absorbing_summary = absorbing_case.absorbing
@assert absorbing_summary.policy_invariance_error <= ABSORBING_INVARIANCE_TOL
@assert absorbing_summary.aggregate_invariance_error <= ABSORBING_INVARIANCE_TOL
@assert absorbing_summary.per_variety_scaling_error <= ABSORBING_INVARIANCE_TOL
@assert all(iszero(b.ν_b_eff) for b in absorbing_bgps)     "The legacy compatibility field must be zero at every algebraically restated absorbing state."

write_csv(joinpath(OUTDIR, "absorbing_normalization_path.csv"), absorbing_rows,
    ["switch_index", "N_US", "N_W", "phi_US", "phi_W",
     "q_US", "d_US", "q_W", "d_W", "Nq_US", "Nd_US", "Nq_W", "Nd_W",
     "Q_US", "Q_W"])

switch_t = [r.switch_index for r in absorbing_rows]
p_us_scale = plot(switch_t, [r.Nq_US for r in absorbing_rows]; lw=2.4,
    label="N_US q_US", xlabel="switch-state index", ylabel="normalized level",
    title="US per-variety 1/N scaling")
plot!(p_us_scale, switch_t, [r.Nd_US for r in absorbing_rows]; lw=2.2, ls=:dash,
      label="N_US d_US")
p_w_scale = plot(switch_t, [r.Nq_W for r in absorbing_rows]; lw=2.4,
    label="N_W q_W", xlabel="switch-state index", ylabel="normalized level",
    title="RoW per-variety 1/N scaling")
plot!(p_w_scale, switch_t, [r.Nd_W for r in absorbing_rows]; lw=2.2, ls=:dash,
      label="N_W d_W")
fig_absorbing_normalization = plot(p_us_scale, p_w_scale; layout=(1, 2), size=(1250, 440))
savefig(fig_absorbing_normalization, joinpath(OUTDIR, "absorbing_normalization_path.png"))
display(fig_absorbing_normalization)


## 5. Full-Path Error and Plot Helpers

Prices, policy variables, capitalization, and theorem leakages are compared in logs. The AHP NFA object is signed and can cross zero, so it is compared in levels using the maximum absolute difference; Notebook 14's invalid \(\log(-NFA)\) diagnostic is not retained.


In [ ]:
function observable_paths(case)
    result = case.result
    T = length(result.u_path)
    q = Float64.(path_vector(result, :q_US))
    d = Float64.(path_vector(result, :d_US))
    phi = Float64.(path_vector(result, :φ_US))
    Q = Float64.(path_vector(result, :Q_US))
    cond_1b = Float64.(result.diagnostics.cond_1b[1:T])
    return (;
        q_US=q,
        d_over_q_US=d ./ q,
        phi_US=phi,
        Q_US=Q,
        rebased_NFA=Float64.(case.nfa.rebased_NFA),
        cond_1b=cond_1b)
end

ERROR_SPECS = [
    (; metric=:log_q_US, path=:q_US, start_t=1, mode=:log,
       max_col=:max_log_q_US_error, t_col=:t_max_log_q_US_error,
       plot_label="log q_US"),
    (; metric=:log_d_over_q_US, path=:d_over_q_US, start_t=1, mode=:log,
       max_col=:max_log_d_over_q_US_error, t_col=:t_max_log_d_over_q_US_error,
       plot_label="log(d/q)_US"),
    (; metric=:log_phi_US, path=:phi_US, start_t=1, mode=:log,
       max_col=:max_log_phi_US_error, t_col=:t_max_log_phi_US_error,
       plot_label="log phi_US"),
    (; metric=:log_Q_US, path=:Q_US, start_t=1, mode=:log,
       max_col=:max_log_Q_US_error, t_col=:t_max_log_Q_US_error,
       plot_label="log Q_US"),
    (; metric=:rebased_NFA, path=:rebased_NFA, start_t=1, mode=:level,
       max_col=:max_abs_rebased_NFA_error, t_col=:t_max_abs_rebased_NFA_error,
       plot_label="(NFA_t-NFA_1)/Y_US,t"),
    (; metric=:log_cond_1b, path=:cond_1b, start_t=2, mode=:log,
       max_col=:max_log_cond_1b_error, t_col=:t_max_log_cond_1b_error,
       plot_label="log switch leakage")]

PATH_SPECS = [
    (; path=:q_US, start_t=1, title="q_US", ylabel="q_US,t"),
    (; path=:d_over_q_US, start_t=1, title="d_US / q_US", ylabel="(d/q)_US,t"),
    (; path=:phi_US, start_t=1, title="phi_US", ylabel="phi_US,t"),
    (; path=:Q_US, start_t=1, title="Q_US", ylabel="Q_US,t"),
    (; path=:rebased_NFA, start_t=1, title="Rebased U.S. NFA",
       ylabel="(NFA_t-NFA_1)/Y_US,t"),
    (; path=:cond_1b, start_t=2, title="Switch leakage", ylabel="switch leakage")]

function missing_error_tuple(domain_status::String, domain_message::String)
    out = (;)
    for spec in ERROR_SPECS
        out = merge(out, NamedTuple{(spec.max_col, spec.t_col)}((missing, missing)))
    end
    return merge(out, (; log_cond_1b_start_t=2,
                       domain_status=domain_status,
                       domain_message=domain_message))
end

function max_path_error_for_values(case_values, ref_values, start_t::Int,
                                   T0::Int, label::String, mode::Symbol)
    T = min(T0, length(case_values), length(ref_values))
    T < start_t && return (value=missing, t=missing, status="empty_window",
                           message="$label has no observations in t=$start_t:$T0")
    idx = collect(start_t:T)
    cv = Float64.(case_values[idx])
    rv = Float64.(ref_values[idx])
    bad = findfirst(i -> !(isfinite(cv[i]) && isfinite(rv[i])), eachindex(idx))
    bad === nothing || return (value=missing, t=idx[bad], status="domain_failure",
        message=@sprintf("%s nonfinite at t=%d (case=%g, ref=%g)",
                         label, idx[bad], cv[bad], rv[bad]))
    if mode == :log
        bad_positive = findfirst(i -> !(cv[i] > 0 && rv[i] > 0), eachindex(idx))
        bad_positive === nothing || return (
            value=missing, t=idx[bad_positive], status="domain_failure",
            message=@sprintf("%s nonpositive at t=%d (case=%g, ref=%g)",
                             label, idx[bad_positive], cv[bad_positive], rv[bad_positive]))
        diffs = abs.(log.(cv) .- log.(rv))
    elseif mode == :level
        diffs = abs.(cv .- rv)
    else
        error("Unknown comparison mode: $mode")
    end
    j = argmax(diffs)
    return (value=diffs[j], t=idx[j], status="ok", message="")
end

function comparison_metrics(case, ref, T0::Int)
    if !residual_safe(case) || !residual_safe(ref)
        return missing_error_tuple("residual_unsafe", "case or reference is not residual-safe")
    end
    T = min(length(case.result.u_path), length(ref.result.u_path))
    T0 <= T || return missing_error_tuple(
        "window_error", "T0=$T0 exceeds available comparison length $T")
    obs, obs_ref = observable_paths(case), observable_paths(ref)
    out = (;)
    messages = String[]
    for spec in ERROR_SPECS
        r = max_path_error_for_values(
            getproperty(obs, spec.path), getproperty(obs_ref, spec.path),
            spec.start_t, T0, String(spec.metric), spec.mode)
        out = merge(out, NamedTuple{(spec.max_col, spec.t_col)}((r.value, r.t)))
        r.status == "ok" || push!(messages, r.message)
    end
    return merge(out, (; log_cond_1b_start_t=2,
        domain_status=isempty(messages) ? "ok" : "domain_failure",
        domain_message=join(messages, " | ")))
end

function metric_column_names()
    cols = String[]
    for spec in ERROR_SPECS
        push!(cols, String(spec.max_col), String(spec.t_col))
    end
    push!(cols, "log_cond_1b_start_t", "domain_status", "domain_message")
    return cols
end

plot_values(rows, col::Symbol) = [
    getproperty(r, col) === missing ? NaN : Float64(getproperty(r, col)) for r in rows]

function plot_error_grid(rows; title_prefix::String, filename::String, label_fn)
    isempty(rows) && return Markdown.parse("_No rows to plot._")
    labels = [label_fn(r) for r in rows]
    x = collect(eachindex(rows))
    p1 = plot(title="$title_prefix: prices and policy", xlabel="case",
              ylabel="max absolute error", xticks=(x, labels),
              xrotation=35, legend=:topleft)
    for spec in ERROR_SPECS[1:3]
        plot!(p1, x, plot_values(rows, spec.max_col), marker=:circle,
              lw=2.1, label=spec.plot_label)
    end
    p2 = plot(title="$title_prefix: aggregates and leakage", xlabel="case",
              ylabel="max absolute error", xticks=(x, labels),
              xrotation=35, legend=:topleft)
    for spec in ERROR_SPECS[4:6]
        plot!(p2, x, plot_values(rows, spec.max_col), marker=:square,
              lw=2.1, label=spec.plot_label)
    end
    fig = plot(p1, p2, layout=(2, 1), size=(1040, 820), bottom_margin=12mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

function plot_path_grid(cases; title_prefix::String, T0::Int,
                        filename::String, label_fn)
    ok_cases = filter(residual_safe, cases)
    isempty(ok_cases) && return Markdown.parse("_No successful cases to plot._")
    panels = Plots.Plot[]
    for spec in PATH_SPECS
        p = plot(title="$title_prefix: $(spec.title)", xlabel="period t",
                 ylabel=spec.ylabel, legend=:outerright)
        for c in ok_cases
            vals = getproperty(observable_paths(c), spec.path)
            T = min(T0, length(vals))
            T < spec.start_t && continue
            tt = spec.start_t:T
            plot!(p, tt, vals[tt], lw=2.0, label=label_fn(c))
        end
        push!(panels, p)
    end
    fig = plot(panels..., layout=(3, 2), size=(1180, 980), bottom_margin=8mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

function path_rows_for_cases(cases; experiment::String, T0::Int)
    rows = NamedTuple[]
    for c in cases
        residual_safe(c) || continue
        obs = observable_paths(c)
        T = min(T0, length(obs.q_US))
        for t in 1:T
            push!(rows, (;
                experiment=experiment, T_report=c.summary.T_report,
                n_buffer=c.summary.n_buffer,
                terminal_rule=c.summary.terminal_rule, t=t,
                q_US=obs.q_US[t], d_over_q_US=obs.d_over_q_US[t],
                phi_US=obs.phi_US[t], Q_US=obs.Q_US[t],
                rebased_NFA=obs.rebased_NFA[t], cond_1b=obs.cond_1b[t]))
        end
    end
    return rows
end

rule_label(c) = c.summary.terminal_rule
row_rule_label(r) = r.terminal_rule

path_cols = ["experiment", "T_report", "n_buffer", "terminal_rule", "t",
             "q_US", "d_over_q_US", "phi_US", "Q_US",
             "rebased_NFA", "cond_1b"]

println("Installed signed-safe full-path metric and plotting helpers.")


## 6. Non-Circular HKT Scalar Entry Gate

The HKT gate is evaluated only on independently solved local-loglinear paths and preserves Notebook 14's conditions: U.S. dominance in labor income and output, declining tail dominance ratios, convergence of the U.S. funding ratio to one, and agreement between the local-loglinear and HKT scalar successor policies. The additional quantities requested here are plotted and exported as diagnostics; they do not change that gate:

$$
\frac{e_{W,t}}{e_{US,t}},
\qquad
\frac{\mathcal Q_{W,t}}{e_{US,t}},
\qquad
\zeta_t=\frac{\mathcal Q_{US,t}^u}{\beta e_{US,t}^u},
$$

$$
\Lambda_t^u,
\qquad
\Lambda_t^b,
\qquad
\frac{\Lambda_t^u}{\Lambda_t^b},
\qquad
\Lambda_t^z:=\frac{R_{A,t}^z}{R_{US,t}^z}.
$$

A return stored in state record $i$ is realized at payoff date $i+1$. The multiplier panels therefore use records $1{:}T-1$, plot them at dates $2{:}T$, and exclude the artificial $T+1$ successor. Notebook period $t=k$ corresponds to model date $k-1$; both indices are exported.


In [ ]:
function tail_log_slope(values)
    idx = [i for i in eachindex(values)
           if values[i] !== missing && isfinite(Float64(values[i])) &&
              Float64(values[i]) > 0]
    length(idx) < 2 && return NaN
    x = Float64.(idx)
    y = log.([Float64(values[i]) for i in idx])
    xbar, ybar = mean(x), mean(y)
    sxx = sum(abs2, x .- xbar)
    sxx <= 0 && return NaN
    return sum((x .- xbar) .* (y .- ybar)) / sxx
end

function aggregate_Q_US(p, phi, N, block)
    return G_N_US(p, phi) * N * block.q
end

function aggregate_Q_W(p, phi, N, block)
    return G_N_W(p, phi) * N * block.q
end

function ll_successor_dominance(case)
    result = case.result
    p = result.params
    N_US_next = terminal_successor_N_US(result)
    N_W_next = terminal_successor_N_W(result)
    phi_US_next = terminal_successor_phi_US(result, :local_loglinear)
    phi_W_next = terminal_successor_phi_W(result, :local_loglinear)
    us_next = us_block(p, :u, phi_US_next, N_US_next)
    rw_next = row_block(p, phi_W_next, N_W_next)
    Q_US_next = aggregate_Q_US(p, phi_US_next, N_US_next, us_next)
    Q_W_next = aggregate_Q_W(p, phi_W_next, N_W_next, rw_next)

    hkt_phi_next = hkt_scalar_phi_us(p, N_US_next)
    us_hkt_next = us_block(p, :u, hkt_phi_next, N_US_next)
    Q_US_hkt_next = aggregate_Q_US(p, hkt_phi_next, N_US_next, us_hkt_next)
    return (;
        successor_e_W_over_e_US=rw_next.e / us_next.e,
        successor_Y_W_over_Y_US=rw_next.Y / us_next.Y,
        successor_Q_W_over_e_US=Q_W_next / us_next.e,
        successor_zeta=Q_US_next / (p.β * us_next.e),
        successor_HKT_e_W_over_e_US=rw_next.e / us_hkt_next.e,
        successor_HKT_Y_W_over_Y_US=rw_next.Y / us_hkt_next.Y,
        successor_HKT_Q_W_over_e_US=Q_W_next / us_hkt_next.e,
        successor_HKT_zeta=Q_US_hkt_next / (p.β * us_hkt_next.e),
        successor_phi_US_LL=phi_US_next,
        successor_phi_US_HKT_scalar=hkt_phi_next,
        successor_HKT_phi_over_floor=hkt_phi_next / p.φ_floor,
        successor_LL_vs_HKT_phi_log_gap=abs(log(phi_US_next) - log(hkt_phi_next)))
end

function hkt_prerequisite_tail_rows(case; last_k::Int=8)
    residual_safe(case) || return NamedTuple[]
    result = case.result
    p = result.params
    u = result.u_path
    T = length(u)
    lo = max(1, T - last_k + 1)
    rows = NamedTuple[]
    for i in lo:T
        s = u[i]
        hkt_phi = hkt_scalar_phi_us(p, s.N_US)
        if i >= 2
            prev = u[i - 1]
            Lambda_u = prev.R_A_u / prev.R_US_u
            Lambda_b = prev.R_A_b / prev.R_US_b
            Lambda_ratio = Lambda_u / Lambda_b
        else
            Lambda_u = missing
            Lambda_b = missing
            Lambda_ratio = missing
        end
        zeta = s.Q_US / (p.β * s.e_US)
        push!(rows, (;
            T_report=T,
            model_T=T - 1,
            n_buffer=case.summary.n_buffer,
            terminal_rule=case.summary.terminal_rule,
            t=i,
            model_t=i - 1,
            periods_to_report_end=T - i,
            e_W_over_e_US=s.e_W / s.e_US,
            Y_W_over_Y_US=s.Y_W / s.Y_US,
            Q_W_over_e_US=s.Q_W / s.e_US,
            Q_US_over_e_W=s.Q_US / s.e_W,
            zeta=zeta,
            abs_zeta_gap=abs(zeta - 1),
            Lambda_u=Lambda_u,
            Lambda_b=Lambda_b,
            Lambda_u_over_Lambda_b=Lambda_ratio,
            phi_US=s.φ_US,
            hkt_scalar_phi_US_same_state=hkt_phi,
            abs_log_phi_US_vs_HKT_scalar=abs(log(s.φ_US) - log(hkt_phi)),
            phi_W=s.φ_W,
            q_US=s.q_US,
            q_W=s.q_W,
            residual_norm=s.residual_norm))
    end
    return rows
end

function hkt_gate_reason(; safe, psi_ok, e_tail, Y_tail, e_next,
                         hkt_e_next, e_slope, funding_gap, phi_gap,
                         hkt_phi, phi_floor)
    reasons = String[]
    safe || push!(reasons, "local-loglinear path is not hard-valid")
    psi_ok || push!(reasons, "local-loglinear effective kernel is not regular")
    finite_number(e_tail) && e_tail <= HKT_E_RATIO_TOL ||
        push!(reasons, "tail e_W/e_US exceeds threshold")
    finite_number(Y_tail) && Y_tail <= HKT_E_RATIO_TOL ||
        push!(reasons, "tail Y_W/Y_US exceeds threshold")
    finite_number(e_next) && e_next <= HKT_E_RATIO_TOL ||
        push!(reasons, "LL-successor e_W/e_US exceeds threshold")
    finite_number(hkt_e_next) && hkt_e_next <= HKT_E_RATIO_TOL ||
        push!(reasons, "HKT-successor e_W/e_US exceeds threshold")
    finite_number(e_slope) && e_slope < 0 ||
        push!(reasons, "tail e_W/e_US is not falling")
    finite_number(funding_gap) && funding_gap <= HKT_FUNDING_LIMIT_GAP_TOL ||
        push!(reasons, "distance from zeta=1 is too large")
    finite_number(phi_gap) && phi_gap <= HKT_SUCCESSOR_PHI_LOG_GAP_TOL ||
        push!(reasons, "LL and HKT successor phi values remain too far apart")
    finite_number(hkt_phi) && phi_floor <= hkt_phi <= 1 - phi_floor ||
        push!(reasons, "HKT successor phi lies outside solver policy bounds")
    return isempty(reasons) ? "passed" : join(reasons, " | ")
end

function hkt_prerequisite_summary(case; last_k::Int=8)
    rows = hkt_prerequisite_tail_rows(case; last_k=last_k)
    if isempty(rows)
        return (;
            T_report=case.summary.T_report, n_buffer=case.summary.n_buffer,
            terminal_rule=case.summary.terminal_rule, status=case.summary.status,
            residual_safe=false, tail_window=last_k, final_t=missing,
            final_e_W_over_e_US=missing, max_tail_e_W_over_e_US=missing,
            tail_log_slope_e_W_over_e_US=missing,
            final_Y_W_over_Y_US=missing, max_tail_Y_W_over_Y_US=missing,
            final_Q_W_over_e_US=missing, max_tail_Q_W_over_e_US=missing,
            tail_log_slope_Q_W_over_e_US=missing,
            final_zeta=missing, max_tail_abs_zeta_gap=missing,
            final_Lambda_u=missing, final_Lambda_b=missing,
            final_Lambda_u_over_Lambda_b=missing,
            max_tail_Lambda_u_over_Lambda_b=missing,
            lambda_diagnostics_regular=false,
            successor_e_W_over_e_US=missing, successor_Y_W_over_Y_US=missing,
            successor_Q_W_over_e_US=missing,
            successor_zeta=missing, successor_HKT_e_W_over_e_US=missing,
            successor_HKT_Y_W_over_Y_US=missing,
            successor_HKT_Q_W_over_e_US=missing, successor_HKT_zeta=missing,
            successor_phi_US_LL=missing, successor_phi_US_HKT_scalar=missing,
            successor_HKT_phi_over_floor=missing,
            successor_LL_vs_HKT_phi_log_gap=missing,
            max_tail_abs_log_phi_US_vs_HKT_scalar=missing,
            max_tail_residual_norm=missing, psi_ok=missing,
            equity_weights_ok=missing, hkt_entry_gate=false,
            hkt_entry_reason="non-hard-valid local-loglinear path")
    end

    final = rows[end]
    succ = ll_successor_dominance(case)
    e_values = [r.e_W_over_e_US for r in rows]
    Y_values = [r.Y_W_over_Y_US for r in rows]
    q_values = [r.Q_W_over_e_US for r in rows]
    lambda_rows = filter(r -> r.Lambda_u !== missing, rows)
    lambda_regular = !isempty(lambda_rows) &&
        all(r -> isfinite(r.Lambda_u) && isfinite(r.Lambda_b) &&
                 isfinite(r.Lambda_u_over_Lambda_b) &&
                 r.Lambda_u > 0 && r.Lambda_b > 0 &&
                 r.Lambda_u_over_Lambda_b > 0, lambda_rows)
    max_tail_e = maximum(e_values)
    max_tail_Y = maximum(Y_values)
    max_tail_q = maximum(q_values)
    e_slope = tail_log_slope(e_values)
    q_slope = tail_log_slope(q_values)
    max_funding_gap = maximum(r.abs_zeta_gap for r in rows)
    safe = residual_safe(case)
    psi_ok = case.summary.psi_ok === true
    hkt_phi_in_bounds = case.result.params.φ_floor <=
        succ.successor_phi_US_HKT_scalar <= 1 - case.result.params.φ_floor
    # Preserve Notebook 14's gate. Q_W/e_US and all Lambda quantities are
    # requested diagnostics and do not enter this Boolean.
    gate = safe && psi_ok &&
           max_tail_e <= HKT_E_RATIO_TOL &&
           max_tail_Y <= HKT_E_RATIO_TOL &&
           succ.successor_e_W_over_e_US <= HKT_E_RATIO_TOL &&
           succ.successor_HKT_e_W_over_e_US <= HKT_E_RATIO_TOL &&
           isfinite(e_slope) && e_slope < 0 &&
           max_funding_gap <= HKT_FUNDING_LIMIT_GAP_TOL &&
           succ.successor_LL_vs_HKT_phi_log_gap <= HKT_SUCCESSOR_PHI_LOG_GAP_TOL &&
           hkt_phi_in_bounds

    reason = hkt_gate_reason(
        safe=safe, psi_ok=psi_ok, e_tail=max_tail_e, Y_tail=max_tail_Y,
        e_next=succ.successor_e_W_over_e_US,
        hkt_e_next=succ.successor_HKT_e_W_over_e_US,
        e_slope=e_slope, funding_gap=max_funding_gap,
        phi_gap=succ.successor_LL_vs_HKT_phi_log_gap,
        hkt_phi=succ.successor_phi_US_HKT_scalar,
        phi_floor=case.result.params.φ_floor)


    return (;
        T_report=case.summary.T_report,
        n_buffer=case.summary.n_buffer,
        terminal_rule=case.summary.terminal_rule,
        status=case.summary.status,
        residual_safe=safe,
        tail_window=last_k,
        final_t=final.t,
        final_e_W_over_e_US=final.e_W_over_e_US,
        max_tail_e_W_over_e_US=max_tail_e,
        tail_log_slope_e_W_over_e_US=e_slope,
        final_Y_W_over_Y_US=final.Y_W_over_Y_US,
        max_tail_Y_W_over_Y_US=max_tail_Y,
        final_Q_W_over_e_US=final.Q_W_over_e_US,
        max_tail_Q_W_over_e_US=max_tail_q,
        tail_log_slope_Q_W_over_e_US=q_slope,
        final_zeta=final.zeta,
        max_tail_abs_zeta_gap=max_funding_gap,
        final_Lambda_u=final.Lambda_u,
        final_Lambda_b=final.Lambda_b,
        final_Lambda_u_over_Lambda_b=final.Lambda_u_over_Lambda_b,
        max_tail_Lambda_u_over_Lambda_b=maximum(
            r.Lambda_u_over_Lambda_b for r in lambda_rows),
        lambda_diagnostics_regular=lambda_regular,
        successor_e_W_over_e_US=succ.successor_e_W_over_e_US,
        successor_Y_W_over_Y_US=succ.successor_Y_W_over_Y_US,
        successor_Q_W_over_e_US=succ.successor_Q_W_over_e_US,
        successor_zeta=succ.successor_zeta,
        successor_HKT_e_W_over_e_US=succ.successor_HKT_e_W_over_e_US,
        successor_HKT_Y_W_over_Y_US=succ.successor_HKT_Y_W_over_Y_US,
        successor_HKT_Q_W_over_e_US=succ.successor_HKT_Q_W_over_e_US,
        successor_HKT_zeta=succ.successor_HKT_zeta,
        successor_phi_US_LL=succ.successor_phi_US_LL,
        successor_phi_US_HKT_scalar=succ.successor_phi_US_HKT_scalar,
        successor_HKT_phi_over_floor=succ.successor_HKT_phi_over_floor,
        successor_LL_vs_HKT_phi_log_gap=succ.successor_LL_vs_HKT_phi_log_gap,
        max_tail_abs_log_phi_US_vs_HKT_scalar=maximum(
            r.abs_log_phi_US_vs_HKT_scalar for r in rows),
        max_tail_residual_norm=maximum(r.residual_norm for r in rows),
        psi_ok=psi_ok,
        equity_weights_ok=case.summary.equity_weights_ok,
        hkt_entry_gate=gate,
        hkt_entry_reason=reason)
end

function plot_hkt_prerequisites(cases; filename::String)
    ok_cases = filter(residual_safe, cases)
    isempty(ok_cases) && return Markdown.parse("_No residual-safe local-loglinear cases to plot._")

    p_e = plot(title="Country income dominance", xlabel="state date t",
               ylabel="e_W / e_US", legend=:outerright)
    p_Q = plot(title="RoW capitalization relative to U.S. income",
               xlabel="state date t", ylabel="Q_W / e_US",
               legend=:outerright)
    p_QUS = plot(title="U.S. capitalization relative to RoW income",
                 xlabel="state date t", ylabel="Q_US / e_W",
                 legend=:outerright)
    p_z = plot(title="U.S. stock-funding ratio", xlabel="state date t",
               ylabel="zeta = Q_US / (beta e_US)",
               legend=:outerright)
    p_lu = plot(title="Continuation portfolio multiplier",
                xlabel="payoff date t", ylabel="Lambda_u",
                legend=:outerright)
    p_lb = plot(title="Switch portfolio multiplier",
                xlabel="payoff date t", ylabel="Lambda_b",
                legend=:outerright)
    p_lr = plot(title="Weak switch-payoff diagnostic",
                xlabel="payoff date t", ylabel="Lambda_u / Lambda_b",
                legend=:outerright)

    for c in ok_cases
        u = c.result.u_path
        T = length(u)
        tt = 1:T
        label = "T=$(c.summary.T_report)"
        plot!(p_e, tt, [s.e_W / s.e_US for s in u], lw=2.0, label=label)
        plot!(p_Q, tt, [s.Q_W / s.e_US for s in u], lw=2.0, label=label)
        plot!(p_QUS, tt, [s.Q_US / s.e_W for s in u], lw=2.0, label=label)
        plot!(p_z, tt, [s.Q_US / (c.result.params.β * s.e_US) for s in u],
              lw=2.0, label=label)
        if T >= 2
            payoff_t = 2:T
            prev = u[1:(T - 1)]
            lambda_u = [s.R_A_u / s.R_US_u for s in prev]
            lambda_b = [s.R_A_b / s.R_US_b for s in prev]
            plot!(p_lu, payoff_t, lambda_u, lw=2.0, label=label)
            plot!(p_lb, payoff_t, lambda_b, lw=2.0, label=label)
            plot!(p_lr, payoff_t, lambda_u ./ lambda_b, lw=2.0, label=label)
        end
    end

    hline!(p_e, [HKT_E_RATIO_TOL], lc=:black, ls=:dash, lw=1.0,
           label="Notebook 14 gate")
    hline!(p_z, [1.0], lc=:black, ls=:dash, lw=1.0, label="HKT target")
    hline!(p_lu, [1.0], lc=:black, ls=:dot, lw=1.0, label="one")
    if CONFIGURED_LAMBDA_BOUND !== nothing
        hline!(p_lr, [CONFIGURED_LAMBDA_BOUND], lc=:black, ls=:dash,
               lw=1.0, label="configured bound")
    end

    fig = plot(p_e, p_Q, p_QUS, p_z, p_lu, p_lb, p_lr,
               layout=@layout([a b; c d; e f; g]),
               size=(1220, 1550), bottom_margin=7mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

hkt_prereq_cases = sort(copy(safe_horizon_cases); by=c -> c.summary.T_report)
hkt_prereq_tail_rows = reduce(vcat,
    [hkt_prerequisite_tail_rows(c; last_k=HKT_PREREQ_TAIL_WINDOW)
     for c in hkt_prereq_cases]; init=NamedTuple[])
hkt_prereq_plot_rows = reduce(vcat,
    [hkt_prerequisite_tail_rows(c; last_k=length(c.result.u_path))
     for c in hkt_prereq_cases]; init=NamedTuple[])
hkt_prereq_summary_rows = [
    hkt_prerequisite_summary(c; last_k=HKT_PREREQ_TAIL_WINDOW)
    for c in hkt_prereq_cases]

hkt_gate_rows = filter(r -> r.hkt_entry_gate, hkt_prereq_summary_rows)
HKT_ENTRY_T_REPORT = isempty(hkt_gate_rows) ? missing :
    minimum(r.T_report for r in hkt_gate_rows)
selected_hkt_gate_summary = only(
    filter(r -> r.T_report == SELECTED_T_REPORT, hkt_prereq_summary_rows))
HKT_GATE_PASSED = selected_hkt_gate_summary.hkt_entry_gate

function representative_cases(cases; max_cases::Int=6)
    length(cases) <= max_cases && return cases
    idx = unique(round.(Int, range(1, length(cases); length=max_cases)))
    return cases[idx]
end
hkt_prereq_plot_cases = representative_cases(hkt_prereq_cases)

hkt_prereq_tail_cols = [
    "T_report", "model_T", "n_buffer", "terminal_rule", "t", "model_t",
    "periods_to_report_end", "e_W_over_e_US", "Y_W_over_Y_US",
    "Q_W_over_e_US", "Q_US_over_e_W", "zeta", "abs_zeta_gap",
    "Lambda_u", "Lambda_b", "Lambda_u_over_Lambda_b",
    "phi_US", "hkt_scalar_phi_US_same_state",
    "abs_log_phi_US_vs_HKT_scalar", "phi_W", "q_US", "q_W", "residual_norm"]

hkt_prereq_summary_cols = [
    "T_report", "n_buffer", "terminal_rule", "status", "residual_safe",
    "tail_window", "final_t", "final_e_W_over_e_US",
    "max_tail_e_W_over_e_US", "tail_log_slope_e_W_over_e_US",
    "final_Y_W_over_Y_US", "max_tail_Y_W_over_Y_US",
    "final_Q_W_over_e_US", "max_tail_Q_W_over_e_US",
    "tail_log_slope_Q_W_over_e_US", "final_zeta",
    "max_tail_abs_zeta_gap", "final_Lambda_u", "final_Lambda_b",
    "final_Lambda_u_over_Lambda_b", "max_tail_Lambda_u_over_Lambda_b",
    "lambda_diagnostics_regular", "successor_e_W_over_e_US",
    "successor_Y_W_over_Y_US", "successor_Q_W_over_e_US",
    "successor_zeta", "successor_HKT_e_W_over_e_US",
    "successor_HKT_Y_W_over_Y_US", "successor_HKT_Q_W_over_e_US", "successor_HKT_zeta",
    "successor_phi_US_LL", "successor_phi_US_HKT_scalar",
    "successor_HKT_phi_over_floor", "successor_LL_vs_HKT_phi_log_gap",
    "max_tail_abs_log_phi_US_vs_HKT_scalar", "max_tail_residual_norm",
    "psi_ok", "equity_weights_ok", "hkt_entry_gate", "hkt_entry_reason"]

write_csv(joinpath(OUTDIR, "hkt_scalar_prerequisite_tail_rows.csv"),
          hkt_prereq_tail_rows, hkt_prereq_tail_cols)
write_csv(joinpath(OUTDIR, "hkt_scalar_prerequisite_plot_data.csv"),
          hkt_prereq_plot_rows, hkt_prereq_tail_cols)
write_csv(joinpath(OUTDIR, "hkt_scalar_prerequisite_summary.csv"),
          hkt_prereq_summary_rows, hkt_prereq_summary_cols)

display(markdown_table(hkt_prereq_summary_rows,
    ["T_report", "max_tail_e_W_over_e_US", "max_tail_Y_W_over_Y_US",
     "max_tail_Q_W_over_e_US",
     "tail_log_slope_e_W_over_e_US", "tail_log_slope_Q_W_over_e_US",
     "max_tail_abs_zeta_gap", "max_tail_Lambda_u_over_Lambda_b",
     "lambda_diagnostics_regular", "successor_LL_vs_HKT_phi_log_gap", "psi_ok",
     "hkt_entry_gate", "hkt_entry_reason"];
    maxrows=length(hkt_prereq_summary_rows)))
println("First explored horizon passing the preserved Notebook 14 HKT gate: ", HKT_ENTRY_T_REPORT)
println("Selected longest LL horizon passes gate: ", HKT_GATE_PASSED,
        " (", selected_hkt_gate_summary.hkt_entry_reason, ")")


In [ ]:
fig_hkt_prereq = plot_hkt_prerequisites(hkt_prereq_plot_cases;
    filename="hkt_scalar_prerequisite_diagnostics.png")
fig_hkt_prereq


## 7. Terminal-Rule Sensitivity at the Selected Full Horizon

The default run evaluates the local-loglinear reference and conditionally the HKT-scalar rule. The HKT rule is attempted only when the selected path passes the independent entry gate, and it is accepted only when the full two-country solution is residual-safe, \(\Psi\) remains regular, and the scalar root is solved to `HKT_SCALAR_ROOT_TOL`. Set `AHP_ZERO_NU_B_EXTENDED_TERMINAL_RULES=true` to add the legacy sensitivity rules.


In [ ]:
terminal_ref = selected_horizon_case
non_hkt_rules = filter(!=(:hkt_scalar_terminal_us), TERMINAL_RULES)
terminal_cases = Any[]
for rule in non_hkt_rules
    if rule == :local_loglinear
        push!(terminal_cases, terminal_ref)
    else
        push!(terminal_cases, run_tail_case(SELECTED_T_REPORT, NO_BUFFER, rule;
                                            branch_iters=terminal_ref.summary.branch_iters,
                                            warm_start_case=terminal_ref))
    end
end

HKT_CASE = nothing
HKT_ADMISSIBLE = false
HKT_ADMISSIBILITY_REASON = HKT_GATE_PASSED ? "not attempted" : selected_hkt_gate_summary.hkt_entry_reason
if HKT_GATE_PASSED
    hkt_outcome = run_residual_safe_case(SELECTED_T_REPORT, NO_BUFFER, :hkt_scalar_terminal_us;
                                         warm_start_case=terminal_ref)
    HKT_CASE = hkt_outcome.case
    scalar_ok = finite_number(HKT_CASE.summary.hkt_scalar_root_residual) &&
                HKT_CASE.summary.hkt_scalar_root_residual <= HKT_SCALAR_ROOT_TOL
    psi_ok = HKT_CASE.status == :ok && HKT_CASE.summary.psi_ok === true
    HKT_ADMISSIBLE = residual_safe(HKT_CASE) && psi_ok && scalar_ok
    reasons = String[]
    residual_safe(HKT_CASE) || push!(reasons, "full two-country HKT-terminal solve is residual-unsafe")
    psi_ok || push!(reasons, "effective-kernel regularity failed")
    scalar_ok || push!(reasons, "HKT scalar terminal root residual exceeds tolerance")
    HKT_ADMISSIBILITY_REASON = isempty(reasons) ? "passed" : join(reasons, " | ")
    push!(terminal_cases, HKT_CASE)
end

terminal_case_summary = [c.summary for c in terminal_cases]
terminal_rows = NamedTuple[]
for c in terminal_cases
    m = comparison_metrics(c, terminal_ref, SELECTED_T_REPORT)
    push!(terminal_rows, merge((;
        T_report=SELECTED_T_REPORT,
        n_buffer=NO_BUFFER,
        terminal_rule=c.summary.terminal_rule,
        reference_rule="local_loglinear",
        T0=SELECTED_T_REPORT,
        status=c.summary.status,
        residual_safe=residual_safe(c),
        max_u_residual=c.summary.max_u_residual,
        max_bgp_residual=c.summary.max_bgp_residual), m))
end

hkt_terminal_decision_rows = [(;
    T_report=SELECTED_T_REPORT,
    hkt_entry_gate=HKT_GATE_PASSED,
    hkt_entry_reason=selected_hkt_gate_summary.hkt_entry_reason,
    hkt_attempted=HKT_CASE !== nothing,
    hkt_residual_safe=HKT_CASE === nothing ? missing : residual_safe(HKT_CASE),
    hkt_scalar_root_residual=HKT_CASE === nothing ? missing : HKT_CASE.summary.hkt_scalar_root_residual,
    psi_ok=HKT_CASE === nothing ? missing : HKT_CASE.summary.psi_ok,
    equity_weights_ok=HKT_CASE === nothing ? missing : HKT_CASE.summary.equity_weights_ok,
    hkt_admissible=HKT_ADMISSIBLE,
    hkt_admissibility_reason=HKT_ADMISSIBILITY_REASON)]
hkt_terminal_decision_cols = ["T_report", "hkt_entry_gate", "hkt_entry_reason", "hkt_attempted",
                              "hkt_residual_safe", "hkt_scalar_root_residual", "psi_ok",
                              "equity_weights_ok", "hkt_admissible", "hkt_admissibility_reason"]

terminal_metric_cols = vcat(["T_report", "n_buffer", "terminal_rule", "reference_rule",
                             "T0", "status", "residual_safe", "max_u_residual", "max_bgp_residual"],
                            metric_column_names())
terminal_path_rows = path_rows_for_cases(terminal_cases;
    experiment="terminal_rule_selected_full_path", T0=SELECTED_T_REPORT)

write_csv(joinpath(OUTDIR, "terminal_rule_case_summary.csv"), terminal_case_summary, summary_cols)
write_csv(joinpath(OUTDIR, "terminal_rule_sensitivity.csv"), terminal_rows, terminal_metric_cols)
write_csv(joinpath(OUTDIR, "terminal_rule_sensitivity_paths.csv"), terminal_path_rows, path_cols)
write_csv(joinpath(OUTDIR, "hkt_scalar_terminal_decision.csv"),
          hkt_terminal_decision_rows, hkt_terminal_decision_cols)

display(markdown_table(terminal_rows,
    ["T_report", "terminal_rule", "status", "residual_safe", "domain_status",
     "max_u_residual", "max_log_q_US_error", "max_log_phi_US_error"];
    maxrows=length(terminal_rows)))
display(markdown_table(hkt_terminal_decision_rows, hkt_terminal_decision_cols;
                       maxrows=length(hkt_terminal_decision_rows)))


In [ ]:
fig_terminal_paths = plot_path_grid(terminal_cases;
    title_prefix="Terminal-rule paths, T=$SELECTED_T_REPORT, no buffer",
    T0=SELECTED_T_REPORT,
    filename="terminal_rule_sensitivity_paths.png",
    label_fn=rule_label)
fig_terminal_paths

In [ ]:
terminal_error_plot_rows = filter(r -> r.status == "ok" && r.residual_safe, terminal_rows)
fig_terminal_errors = plot_error_grid(terminal_error_plot_rows;
    title_prefix="Terminal-rule errors, T=$SELECTED_T_REPORT, no buffer",
    filename="terminal_rule_sensitivity_errors.png",
    label_fn=row_rule_label)
fig_terminal_errors


## 8. Fundamental-Value-Free Bubble-Share Bounds and Horizon Convergence

Notebook 14 imposed a terminal fundamental value and recursively calculated $1-v_t/q_t$. This section never calculates $v_t$.

To keep terminal-closure error out of the finite product, the notebook fixes a pre-terminal cutoff $K$ measured back from the **selected** horizon by `AHP_ZERO_NU_B_BOUND_ENDPOINT_GUARD`. It uses the numerical continuation $K+1,\ldots,T-1$ only to test the proposed HKT envelopes. The endpoint $T$ and its artificial successor never enter the finite product or those pointwise checks.

Sharing one cutoff across every solved horizon, as in Notebook 14, pins $K$ to the *shortest* solve, which is exactly where the analytic envelope does not apply. Two things break there and both are fixed here:

- **The cutoff.** The theory note requires $K$ chosen so that $C_\varphi N_K^{-\kappa_D}<1$; only then is the knowledge-growth floor $G$ analytic rather than an observed fallback. The cutoff is therefore measured back from the selected zero-$\nu_b$ horizon, and `analytic_growth_floor_ok` reports whether the newly solved counterfactual path actually satisfies this hypothesis. No numerical conclusion is inherited from the common-growth run.
- **The uniform switch exponent.** The fixed primitive $\nu_b=0$ is itself the global future-uniform exponent bound. No switch-specific exponent is constructed or estimated; the normalized-policy and $1/N$ scaling invariants are checked separately.

For exact one-period leakage $a_s$, define

$$
P_{t,K}
:=
\prod_{s=t+1}^{K}\frac{1}{1+a_s}.
$$

If the analytic HKT continuation bounds all leakage after $K$ by $\overline A_K$, then

$$
\boxed{
P_{t,K}e^{-\overline A_K}
\le
\frac{B_{US,t}^u}{q_{US,t}^u}
\le
P_{t,K}.
}
$$

The lower endpoint uses

$$
\overline A_K
=
\frac{C_DN_K^{-\kappa_D}}{G^{\kappa_D}-1}
+
\frac{1-\pi}{\pi}
\frac{C_SN_K^{-\kappa_S}}{G^{\kappa_S}-1},
$$

with the uniform, continuation-indexed constants derived in Logistics/Notes_CJP/On_theoretical_lwbd_bubble_share.md. The upper endpoint additionally needs $a_s\ge0$ for every $s>K$, which is verified only on the solved continuation.

Because a large $K$ leaves a short endpoint guard, the reported prefix product is the object most exposed to terminal-closure error. `bubble_bound_prefix_stability.csv` therefore compares $\log P_{1,K}$ itself — not only its inputs — against every shorter hard-valid horizon that reaches $K$, and `prefix_product_log_gap` enters the `prefix_stable` flag.

The fixed global bound $\nu_b=0$, observed maxima of $\Lambda_t^u/\Lambda_t^b$, and the derived $\underline\zeta$ and $\epsilon_q$ do not by themselves prove every infinite-future envelope premise. The lower endpoint is therefore labelled conditional unless AHP\_ZERO\_NU\_B\_FUTURE\_ENVELOPE\_CERTIFIED=true represents an independent proof of the remaining future premises. If the independent Notebook 14 gate fails, the plotted interval is explicitly labelled diagnostic; if the finite-envelope diagnostics themselves fail, that strictly worse outcome is reported in its own right rather than as a gate failure. `omitted_tail_numerical_estimate` and `envelope_slack_factor` report how far $\overline A_K$ sits above the leakage the solved path actually produces.

Notebook period $t=k$ corresponds to model date $k-1$; both indices are exported.


In [ ]:
function reconstruct_leakage(result::ProductionSimulationResult)
    p = result.params
    u = result.u_path
    bgp = result.bgp_seq
    T = length(u)
    a = zeros(Float64, T)
    dividend = zeros(Float64, T)
    switch = zeros(Float64, T)
    for k in 2:T
        prev = u[k - 1]
        current = u[k]
        b = bgp[k]
        D = current.d_US / current.q_US
        Lambda_u = prev.R_A_u / prev.R_US_u
        Lambda_b = prev.R_A_b / prev.R_US_b
        relative_switch_payoff = (b.q_US + b.d_US) / current.q_US
        L = (Lambda_u / Lambda_b)^p.γ *
            (1 + D)^p.γ *
            relative_switch_payoff^(1 - p.γ)
        dividend[k] = D
        switch[k] = L
        a[k] = D + (1 - p.π_persist) / p.π_persist * L
    end
    return (; a, dividend, switch)
end

# The cutoff K now comes from the SELECTED path. Sharing one cutoff across every
# solved horizon pinned K to the shortest solve (K = 22), which is exactly where
# the analytic envelope does not apply: N_K was 1.34, so C_phi*N_K^{-kappa_D} > 1
# and the knowledge-growth floor silently fell back to an observed number. A
# large K raises N_K and restores the analytic floor. The price is a short
# endpoint guard, so the prefix through K is
# cross-checked against every shorter hard-valid horizon that reaches K.
SELECTED_SOLVE_T = length(selected_horizon_case.result.u_path)
DEFAULT_BOUND_CUTOFF = SELECTED_SOLVE_T - BOUND_ENDPOINT_GUARD
BOUND_CUTOFF = CONFIGURED_BOUND_CUTOFF === nothing ?
    DEFAULT_BOUND_CUTOFF : CONFIGURED_BOUND_CUTOFF
@assert 2 <= BOUND_CUTOFF <= SELECTED_SOLVE_T - 2 "bound cutoff must leave at least two pre-endpoint continuation dates on the selected path"

PREFIX_CHECK_CASES = filter(
    c -> length(c.result.u_path) >= BOUND_CUTOFF, safe_horizon_cases)
BOUND_CAPABLE_CASES = filter(
    c -> length(c.result.u_path) >= BOUND_CUTOFF + 2, safe_horizon_cases)
@assert !isempty(BOUND_CAPABLE_CASES) "no hard-valid horizon is long enough for the configured cutoff"

prefix_log_product(case, K::Int) =
    -sum(log1p(Float64(case.result.diagnostics.a_t[k])) for k in 2:K)

function prefix_stability(case, reference_case, K::Int;
                          comparison::String="vs_selected")
    u = case.result.u_path
    ref = reference_case.result.u_path
    length(u) >= K && length(ref) >= K || error("prefix comparison exceeds solved path")
    a = Float64.(case.result.diagnostics.a_t[1:K])
    ar = Float64.(reference_case.result.diagnostics.a_t[1:K])
    phi_gap = maximum(abs(log(u[t].φ_US) - log(ref[t].φ_US)) for t in 1:K)
    q_gap = maximum(abs(log(u[t].q_US) - log(ref[t].q_US)) for t in 1:K)
    leakage_gap = maximum(abs.(a .- ar))
    nfa_gap = maximum(abs(case.nfa.rebased_NFA[t] -
                          reference_case.nfa.rebased_NFA[t]) for t in 1:K)
    # With a short endpoint guard the reported object is the prefix product
    # itself, so compare that directly rather than only its inputs.
    product_gap = abs(prefix_log_product(case, K) -
                      prefix_log_product(reference_case, K))
    # Gate on the quantities the reported interval is actually built from: the
    # prefix product and the one-period leakages that enter it. The state paths
    # are reported alongside but do not gate, because at a short guard they pick
    # up terminal motion at late dates that moves P by orders of magnitude less
    # than the interval width. Both flags are exported for every comparison.
    product_stable = product_gap <= BOUND_PREFIX_STABILITY_TOL &&
                     leakage_gap <= BOUND_PREFIX_STABILITY_TOL
    paths_stable = phi_gap <= BOUND_PREFIX_STABILITY_TOL &&
                   q_gap <= BOUND_PREFIX_STABILITY_TOL &&
                   nfa_gap <= BOUND_PREFIX_STABILITY_TOL
    return (;
        T_report=case.summary.T_report,
        terminal_rule=case.summary.terminal_rule,
        cutoff_t=K,
        cutoff_model_t=K - 1,
        comparison=comparison,
        reference_T_report=reference_case.summary.T_report,
        endpoint_guard=length(u) - K,
        max_log_phi_US_gap=phi_gap,
        max_log_q_US_gap=q_gap,
        max_leakage_gap=leakage_gap,
        max_rebased_nfa_gap=nfa_gap,
        prefix_product_log_gap=product_gap,
        stability_tolerance=BOUND_PREFIX_STABILITY_TOL,
        prefix_product_stable=product_stable,
        prefix_paths_stable=paths_stable,
        prefix_stable=product_stable)
end

# Comparing the selected path against itself proves nothing, so the bound is
# checked against the longest OTHER horizon that reaches K and still keeps a
# guard of its own. A path whose endpoint is exactly K is not a fair yardstick.
function longest_other_reaching(case, K::Int)
    partners = filter(c -> c.summary.T_report != case.summary.T_report &&
                           length(c.result.u_path) >= K + 1, PREFIX_CHECK_CASES)
    isempty(partners) && return nothing
    return partners[argmax([c.summary.T_report for c in partners])]
end

function stability_for(case)
    partner = longest_other_reaching(case, BOUND_CUTOFF)
    partner === nothing &&
        return prefix_stability(case, selected_horizon_case, BOUND_CUTOFF;
                                comparison="vs_selected_no_partner_available")
    return prefix_stability(case, partner, BOUND_CUTOFF;
                            comparison="vs_longest_other_horizon")
end

prefix_stability_rows = vcat(
    [prefix_stability(c, selected_horizon_case, BOUND_CUTOFF)
     for c in PREFIX_CHECK_CASES],
    [stability_for(c) for c in
     filter(c -> length(c.result.u_path) >= BOUND_CUTOFF + 2, PREFIX_CHECK_CASES)])
prefix_stability_cols = String.(propertynames(first(prefix_stability_rows)))
write_csv(joinpath(OUTDIR, "bubble_bound_prefix_stability.csv"),
          prefix_stability_rows, prefix_stability_cols)

function bubble_bound_data(case, gate_source_case, stability)
    residual_safe(case) || error("Bubble bounds require a hard-valid case")
    result = case.result
    p = result.params
    u = result.u_path
    T = length(u)
    K = BOUND_CUTOFF
    T >= K + 2 || error("Solved horizon must leave a pre-terminal validation segment")

    exact = Float64.(result.diagnostics.a_t[1:T])
    rebuilt = reconstruct_leakage(result)
    leakage_reconstruction_error = maximum(abs.(exact .- rebuilt.a))
    finite_prefix_nonnegative = all(x -> isfinite(x) && x >= 0, exact[1:K])

    # Validate envelopes on the numerical continuation, but never multiply
    # those leakages into P. Exclude endpoint T, which is terminal-adjacent.
    continuation_dates = collect((K + 1):(T - 1))
    state_check_dates = collect(K:(T - 1))
    continuation_nonnegative = all(x -> isfinite(x) && x >= 0,
                                   exact[continuation_dates])

    # ---- cone inputs: zeta lower bound and the q^u price haircut ----------
    zeta_continuation_min = minimum(
        u[k].Q_US / (p.β * u[k].e_US) for k in state_check_dates)
    zeta_lower = CONFIGURED_ZETA_LOWER === nothing ?
        (1 - BOUND_CONE_MARGIN) * zeta_continuation_min : CONFIGURED_ZETA_LOWER
    zeta_lower_source = CONFIGURED_ZETA_LOWER === nothing ?
        "continuation_min_with_margin_conditional" : "environment"
    zeta_lower_ok = zeta_continuation_min >= zeta_lower

    qbar_u = p.β * p.A_L_US_u * p.L_US *
        (1 - p.α_US)^(-1 / (p.ρ_US - 1)) /
        (1 + p.a_US * p.H_US * (1 - p.β))
    q_ratio_continuation_min = minimum(
        u[k].q_US / (qbar_u * u[k].N_US^(p.ν_u - 1)) for k in state_check_dates)
    q_price_epsilon = CONFIGURED_Q_PRICE_EPSILON === nothing ?
        clamp(1 - (1 - BOUND_CONE_MARGIN) * q_ratio_continuation_min,
              1e-6, 1 - 1e-6) : CONFIGURED_Q_PRICE_EPSILON
    q_price_epsilon_source = CONFIGURED_Q_PRICE_EPSILON === nothing ?
        "continuation_min_with_margin_conditional" : "environment"
    q_lower_continuation_ok = all(
        u[k].q_US >= (1 - q_price_epsilon) *
            qbar_u * u[k].N_US^(p.ν_u - 1) for k in state_check_dates)

    lambda_ratio_continuation = Float64[
        (u[k - 1].R_A_u / u[k - 1].R_US_u) /
        (u[k - 1].R_A_b / u[k - 1].R_US_b) for k in continuation_dates]
    observed_lambda_max = maximum(lambda_ratio_continuation)
    lambda_bound = CONFIGURED_LAMBDA_BOUND === nothing ?
        1.05 * observed_lambda_max : CONFIGURED_LAMBDA_BOUND
    lambda_bound_source = CONFIGURED_LAMBDA_BOUND === nothing ?
        "observed_continuation_plus_5pct_conditional" :
        "environment_future_uniform_bound"
    lambda_observed_within_bound = isfinite(lambda_bound) &&
        lambda_bound >= observed_lambda_max

    # The specialization fixes the global exponent bound directly. It is not
    # inferred from a switch date, absorbing state, or fitted continuation path.
    p.ν_b == ZERO_NU_B || error("bubble bound requires the primitive nu_b=0 specialization")
    nu_b_bound = p.ν_b
    nu_b_bound_source = "fixed_primitive_p_nu_b"
    nu_b_ordering = 0 <= nu_b_bound < p.ν_u

    psi = (p.ξ_u - p.ν_u) * (p.ρ_US - 1)
    kappa_D = psi / p.ρ_US
    kappa_S = (p.ν_u - nu_b_bound) * (1 - p.γ)
    C0 = 1 / (p.a_US * p.H_US) + 1
    K_u = (1 - p.α_US) / (p.ϑ_US * p.α_US) *
          ((p.A_X_US_u * p.H_US) /
           (p.A_L_US_u * p.L_US))^(p.ρ_US - 1)
    C_phi = ((C0 - p.β * zeta_lower) /
             (p.β * zeta_lower * K_u))^(1 / p.ρ_US)
    C_D = p.a_US * (1 - p.ϑ_US) / p.ϑ_US * p.H_US * C_phi

    C_b = p.ϑ_US * p.α_US^(-1 / (p.ρ_US - 1)) *
        p.Abar_X_US / p.a_US *
        (1 + p.a_US * (1 - p.ϑ_US) / p.ϑ_US * p.H_US)

    N_K = u[K].N_US
    phi_upper_K = C_phi * N_K^(-kappa_D)
    theoretical_G_lower = 1 + p.a_US * p.H_US * (1 - phi_upper_K)
    observed_G_continuation_min = minimum(
        G_N_US(p, u[k].φ_US) for k in state_check_dates)
    # The note requires K chosen so that C_phi*N_K^{-kappa_D} < 1; only then is
    # the growth floor analytic. Record whether that hypothesis actually holds.
    analytic_growth_floor_ok = phi_upper_K < 1 && theoretical_G_lower > 1
    G_lower = theoretical_G_lower
    G_lower_source = "analytic_scalar_phi_bound"

    C_S = lambda_bound^p.γ *
        (1 + C_D * N_K^(-kappa_D))^p.γ *
        (C_b / ((1 - q_price_epsilon) * qbar_u))^(1 - p.γ)

    denominators_ok = isfinite(G_lower) && G_lower > 1 &&
        isfinite(kappa_D) && kappa_D > 0 &&
        isfinite(kappa_S) && kappa_S > 0
    Abar_K = denominators_ok ?
        C_D * N_K^(-kappa_D) / (G_lower^kappa_D - 1) +
        (1 - p.π_persist) / p.π_persist *
        C_S * N_K^(-kappa_S) / (G_lower^kappa_S - 1) : Inf

    # Numerical yardstick for how slack the analytic envelope is: the solved
    # leakages past K plus a geometric extension of the last observed ratio.
    # A diagnostic only — it is not a bound and never enters any interval.
    ratio_lo = max(2, T - 10)
    tail_ratio = T - 1 > ratio_lo ? exact[T - 1] / exact[T - 2] : NaN
    omitted_tail_numerical = (isfinite(tail_ratio) && 0 < tail_ratio < 1) ?
        sum(exact[continuation_dates]) +
        exact[T - 1] * tail_ratio / (1 - tail_ratio) : NaN

    dividend_envelope = Float64[C_D * s.N_US^(-kappa_D) for s in u]
    switch_envelope = Float64[C_S * s.N_US^(-kappa_S) for s in u]
    tail_dividend_envelope_ok = all(
        rebuilt.dividend[k] <= dividend_envelope[k] * (1 + 1e-10)
        for k in continuation_dates)
    tail_switch_envelope_ok = all(
        rebuilt.switch[k] <= switch_envelope[k] * (1 + 1e-10)
        for k in continuation_dates)

    gate_summary = hkt_prerequisite_summary(
        gate_source_case; last_k=HKT_PREREQ_TAIL_WINDOW)
    gate = gate_summary.hkt_entry_gate
    finite_prefix_valid = finite_prefix_nonnegative &&
        leakage_reconstruction_error <= 1e-8
    finite_envelope_diagnostics_pass = finite_prefix_valid &&
        analytic_growth_floor_ok && continuation_nonnegative && stability.prefix_stable &&
        lambda_observed_within_bound && nu_b_ordering &&
        denominators_ok && isfinite(Abar_K) && Abar_K >= 0 &&
        zeta_lower_ok && q_lower_continuation_ok &&
        tail_dividend_envelope_ok && tail_switch_envelope_ok
    # nu_b=0 is already a global primitive; only the lambda envelope remains
    # an externally configured future-uniform input.
    future_uniform_inputs_configured = CONFIGURED_LAMBDA_BOUND !== nothing
    certified = finite_envelope_diagnostics_pass && gate &&
        FUTURE_ENVELOPE_CERTIFIED

    status = !finite_envelope_diagnostics_pass ?
        "invalid_finite_envelope_diagnostics" :
        !gate ? "diagnostic_only_hkt_gate_failed" :
        certified ? "certified_configured_future_envelope" :
        "conditional_on_unproven_future_envelope"

    logP = zeros(Float64, K)
    for k in K:-1:2
        logP[k - 1] = logP[k] - log1p(exact[k])
    end

    rows = NamedTuple[]
    for t in 1:K
        upper = exp(logP[t])
        lower = finite_envelope_diagnostics_pass ? exp(logP[t] - Abar_K) : missing
        Q = u[t].Q_US
        NFA = case.nfa.NFA[t]
        push!(rows, (;
            T_report=T,
            model_T=T - 1,
            terminal_rule=case.summary.terminal_rule,
            gate_source_terminal_rule=gate_source_case.summary.terminal_rule,
            cutoff_t=K,
            cutoff_model_t=K - 1,
            t=t,
            model_t=t - 1,
            evaluation_date=t in BOUND_EVALUATION_DATES,
            bound_status=status,
            lower_bound_certified=certified,
            hkt_gate_passed=gate,
            prefix_stable=stability.prefix_stable,
            finite_prefix_product=upper,
            omitted_tail_bound=finite_envelope_diagnostics_pass ? Abar_K : missing,
            bubble_share_lower=lower,
            bubble_share_upper=upper,
            relative_interval_width_bound=finite_envelope_diagnostics_pass ?
                1 - exp(-Abar_K) : missing,
            leakage=exact[t],
            dividend_leakage=rebuilt.dividend[t],
            switch_leakage=rebuilt.switch[t],
            dividend_envelope=dividend_envelope[t],
            switch_envelope=switch_envelope[t],
            Q_US=Q,
            NFA=NFA,
            B_agg_lower=lower === missing ? missing : lower * Q,
            B_agg_upper=upper * Q,
            V_agg_lower=(1 - upper) * Q,
            V_agg_upper=lower === missing ? missing : (1 - lower) * Q,
            NFA_bubble_lower=-upper * Q,
            NFA_bubble_upper=lower === missing ? missing : -lower * Q,
            NFA_fund_lower=lower === missing ? missing : NFA + lower * Q,
            NFA_fund_upper=NFA + upper * Q))
    end

    # Report the envelope check over a wider window than the binding one, so a
    # short continuation segment still leaves a legible diagnostic.
    envelope_report_dates = collect(max(2, K - 40):(T - 1))
    envelope_rows = [(;
        T_report=T,
        model_T=T - 1,
        terminal_rule=case.summary.terminal_rule,
        cutoff_t=K,
        cutoff_model_t=K - 1,
        t=k,
        model_t=k - 1,
        binding_continuation_date=k in continuation_dates,
        exact_leakage=exact[k],
        dividend_leakage=rebuilt.dividend[k],
        switch_leakage=rebuilt.switch[k],
        dividend_envelope=dividend_envelope[k],
        switch_envelope=switch_envelope[k],
        total_leakage_envelope=dividend_envelope[k] +
            (1 - p.π_persist) / p.π_persist * switch_envelope[k],
        dividend_envelope_ok=rebuilt.dividend[k] <=
            dividend_envelope[k] * (1 + 1e-10),
        switch_envelope_ok=rebuilt.switch[k] <=
            switch_envelope[k] * (1 + 1e-10))
        for k in envelope_report_dates]

    summary = (;
        T_report=T,
        model_T=T - 1,
        terminal_rule=case.summary.terminal_rule,
        gate_source_terminal_rule=gate_source_case.summary.terminal_rule,
        cutoff_t=K,
        cutoff_model_t=K - 1,
        endpoint_guard=T - K,
        n_continuation_dates=length(continuation_dates),
        bound_status=status,
        lower_bound_certified=certified,
        hkt_gate_passed=gate,
        finite_envelope_diagnostics_pass=finite_envelope_diagnostics_pass,
        future_uniform_inputs_configured=future_uniform_inputs_configured,
        future_envelope_certification_flag=FUTURE_ENVELOPE_CERTIFIED,
        prefix_stable=stability.prefix_stable,
        prefix_paths_stable=stability.prefix_paths_stable,
        prefix_comparison=stability.comparison,
        prefix_reference_T_report=stability.reference_T_report,
        prefix_product_log_gap=stability.prefix_product_log_gap,
        prefix_max_log_q_US_gap=stability.max_log_q_US_gap,
        leakage_reconstruction_error=leakage_reconstruction_error,
        finite_prefix_nonnegative=finite_prefix_nonnegative,
        continuation_nonnegative=continuation_nonnegative,
        validation_start_t=first(continuation_dates),
        validation_end_t=last(continuation_dates),
        zeta_continuation_min=zeta_continuation_min,
        zeta_lower_assumption=zeta_lower,
        zeta_lower_source=zeta_lower_source,
        zeta_lower_ok=zeta_lower_ok,
        q_ratio_continuation_min=q_ratio_continuation_min,
        q_price_epsilon=q_price_epsilon,
        q_price_epsilon_source=q_price_epsilon_source,
        q_lower_continuation_ok=q_lower_continuation_ok,
        observed_lambda_ratio_max=observed_lambda_max,
        lambda_ratio_bound=lambda_bound,
        lambda_bound_source=lambda_bound_source,
        lambda_observed_within_bound=lambda_observed_within_bound,
        nu_b_uniform_bound=nu_b_bound,
        nu_b_bound_source=nu_b_bound_source,
        nu_b_ordering=nu_b_ordering,
        psi=psi, kappa_D=kappa_D, kappa_S=kappa_S,
        K_u=K_u, C_phi=C_phi, C_D=C_D, qbar_u=qbar_u,
        C_b=C_b, C_S=C_S,
        N_K=N_K, phi_upper_K=phi_upper_K,
        analytic_growth_floor_ok=analytic_growth_floor_ok,
        theoretical_G_lower=theoretical_G_lower,
        observed_G_continuation_min=observed_G_continuation_min,
        G_lower=G_lower,
        G_lower_source=G_lower_source,
        omitted_tail_bound=finite_envelope_diagnostics_pass ? Abar_K : missing,
        omitted_tail_numerical_estimate=omitted_tail_numerical,
        envelope_slack_factor=isfinite(omitted_tail_numerical) &&
            omitted_tail_numerical > 0 && isfinite(Abar_K) ?
            Abar_K / omitted_tail_numerical : missing,
        tail_dividend_envelope_ok=tail_dividend_envelope_ok,
        tail_switch_envelope_ok=tail_switch_envelope_ok)
    return (; rows, envelope_rows, summary)
end

bound_specs = [(; case=c, gate_case=c) for c in BOUND_CAPABLE_CASES]
if HKT_ADMISSIBLE && HKT_CASE !== nothing &&
   length(HKT_CASE.result.u_path) >= BOUND_CUTOFF + 2
    push!(bound_specs, (; case=HKT_CASE, gate_case=selected_horizon_case))
end

bound_results = [
    bubble_bound_data(spec.case, spec.gate_case, stability_for(spec.case))
    for spec in bound_specs]
theoretical_bubble_share_rows = reduce(
    vcat, [x.rows for x in bound_results]; init=NamedTuple[])
theoretical_tail_envelope_rows = reduce(
    vcat, [x.envelope_rows for x in bound_results]; init=NamedTuple[])
theoretical_tail_summary_rows = [x.summary for x in bound_results]
selected_theoretical_bubble_rows = filter(
    r -> r.T_report == SELECTED_T_REPORT &&
         r.terminal_rule == selected_horizon_case.summary.terminal_rule,
    theoretical_bubble_share_rows)
selected_theoretical_tail_envelope_rows = filter(
    r -> r.T_report == SELECTED_T_REPORT &&
         r.terminal_rule == selected_horizon_case.summary.terminal_rule,
    theoretical_tail_envelope_rows)
bound_horizon_rows = filter(
    r -> r.evaluation_date && r.terminal_rule == "local_loglinear",
    theoretical_bubble_share_rows)

bubble_bound_cols = [
    "T_report", "model_T", "terminal_rule", "gate_source_terminal_rule",
    "cutoff_t", "cutoff_model_t", "t", "model_t", "evaluation_date",
    "bound_status", "lower_bound_certified", "hkt_gate_passed", "prefix_stable",
    "finite_prefix_product", "omitted_tail_bound",
    "bubble_share_lower", "bubble_share_upper", "relative_interval_width_bound",
    "leakage", "dividend_leakage", "switch_leakage",
    "dividend_envelope", "switch_envelope", "Q_US", "NFA",
    "B_agg_lower", "B_agg_upper", "V_agg_lower", "V_agg_upper",
    "NFA_bubble_lower", "NFA_bubble_upper",
    "NFA_fund_lower", "NFA_fund_upper"]

envelope_cols = [
    "T_report", "model_T", "terminal_rule", "cutoff_t", "cutoff_model_t",
    "t", "model_t", "binding_continuation_date",
    "exact_leakage", "dividend_leakage", "switch_leakage",
    "dividend_envelope", "switch_envelope", "total_leakage_envelope",
    "dividend_envelope_ok", "switch_envelope_ok"]

tail_summary_cols = String.(propertynames(first(theoretical_tail_summary_rows)))

write_csv(joinpath(OUTDIR, "theoretical_bubble_share_bounds.csv"),
          theoretical_bubble_share_rows, bubble_bound_cols)
write_csv(joinpath(OUTDIR, "theoretical_bubble_share_horizon_convergence.csv"),
          bound_horizon_rows, bubble_bound_cols)
write_csv(joinpath(OUTDIR, "theoretical_tail_envelope_check.csv"),
          theoretical_tail_envelope_rows, envelope_cols)
write_csv(joinpath(OUTDIR, "theoretical_tail_envelope_summary.csv"),
          theoretical_tail_summary_rows, tail_summary_cols)

println("Selected solved horizon T:      ", SELECTED_SOLVE_T)
println("Bound cutoff K:                 ", BOUND_CUTOFF,
        " (endpoint guard ", SELECTED_SOLVE_T - BOUND_CUTOFF, ")")
println("Horizons reaching K:            ",
        [c.summary.T_report for c in PREFIX_CHECK_CASES])
println("Horizons carrying a full bound: ",
        [c.summary.T_report for c in BOUND_CAPABLE_CASES])
println("Max prefix-product log gap:     ",
        maximum(r.prefix_product_log_gap for r in prefix_stability_rows))
for r in filter(r -> r.comparison == "vs_longest_other_horizon", prefix_stability_rows)
    @printf("Bound path T=%d vs T=%d at K=%d: logP gap %.3e, leakage gap %.3e, log q gap %.3e (tol %.1e) -> product_stable=%s paths_stable=%s\n",
            r.T_report, r.reference_T_report, r.cutoff_t, r.prefix_product_log_gap,
            r.max_leakage_gap, r.max_log_q_US_gap, r.stability_tolerance,
            r.prefix_product_stable, r.prefix_paths_stable)
end
println("Underflow-safe evaluation-date bounds are displayed and exported in Section 8.1.")
display(markdown_table(theoretical_tail_summary_rows,
    ["T_report", "terminal_rule", "cutoff_t", "endpoint_guard",
     "n_continuation_dates", "bound_status", "lower_bound_certified",
     "hkt_gate_passed", "prefix_comparison", "prefix_reference_T_report",
     "prefix_stable", "prefix_paths_stable", "prefix_product_log_gap",
     "prefix_max_log_q_US_gap",
     "zeta_continuation_min", "zeta_lower_assumption",
     "q_ratio_continuation_min", "q_price_epsilon",
     "nu_b_uniform_bound", "nu_b_bound_source",
     "kappa_D", "kappa_S", "N_K", "phi_upper_K",
     "analytic_growth_floor_ok", "G_lower", "G_lower_source",
     "omitted_tail_bound", "omitted_tail_numerical_estimate",
     "envelope_slack_factor", "tail_dividend_envelope_ok",
     "tail_switch_envelope_ok"];
    maxrows=length(theoretical_tail_summary_rows)))

In [ ]:
function plot_tail_leakage_envelope(rows; filename::String)
    isempty(rows) && return Markdown.parse("_No continuation-envelope rows to plot._")
    tt = [r.t for r in rows]
    exact = [r.exact_leakage for r in rows]
    envelope = [r.total_leakage_envelope for r in rows]
    binding = [r.binding_continuation_date for r in rows]
    p = plot(tt, exact, marker=:circle, lw=2.2, yscale=:log10,
             label="exact solved leakage",
             title="DIAGNOSTIC continuation-envelope check\nendpoint T excluded",
             xlabel="notebook period t (model date t-1)", ylabel="a_t")
    plot!(p, tt, envelope, marker=:diamond, ls=:dash, lw=2.2,
          label="analytic upper envelope")
    if any(binding)
        vspan!(p, [minimum(tt[binding]), maximum(tt[binding])],
               color=:gray70, alpha=0.25, label="binding continuation window")
    end
    fig = plot(p, size=(1000, 590), bottom_margin=8mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

fig_tail_envelope = plot_tail_leakage_envelope(
    selected_theoretical_tail_envelope_rows;
    filename="theoretical_tail_leakage_envelope.png")
display(fig_tail_envelope)

## 8.1 Underflow-Safe Bound Export, Provenance, and Output Files

Calibration and zero-nu_b comparison to the source AHP path:

- `calibration_manifest.csv`, `run_manifest.csv`
- `ahp_nfa_zero_nu_b_comparison_path.csv`, `ahp_nfa_zero_nu_b_comparison_summary.csv`
- `ahp_nfa_zero_nu_b_comparison.png`
- `absorbing_normalization_path.csv`, `absorbing_normalization_path.png`

Hard-valid horizon and Section 6 diagnostics:

- `residual_safe_horizon_progress.csv`, `residual_safe_horizon_attempts.csv`
- `residual_safe_horizon_search.csv`, `selected_horizon_summary.csv`
- `selected_horizon_cold_verification.csv`
- `hkt_scalar_prerequisite_summary.csv`
- `hkt_scalar_prerequisite_tail_rows.csv`
- `hkt_scalar_prerequisite_plot_data.csv`
- `hkt_scalar_prerequisite_diagnostics.png`
- `hkt_scalar_terminal_decision.csv`

Terminal sensitivity:

- `terminal_rule_case_summary.csv`, `terminal_rule_sensitivity.csv`
- `terminal_rule_sensitivity_paths.csv`
- `terminal_rule_sensitivity_paths.png`, `terminal_rule_sensitivity_errors.png`

Pre-terminal theoretical bubble intervals:

- `bubble_bound_prefix_stability.csv`
- `bubble_bound_prefix_product_stability.png`
- `theoretical_bubble_share_bounds.csv`
- `theoretical_bubble_share_horizon_convergence.csv`
- `theoretical_tail_envelope_summary.csv`
- `theoretical_tail_envelope_check.csv`
- `theoretical_bubble_share_bounds.png`
- `theoretical_bubble_share_horizon_convergence.png`
- `theoretical_tail_leakage_envelope.png`

No `fundamental_value_*`, terminal-recursive bubble-share, or terminal-anchor files are produced.


In [ ]:
# Finalize the Section 8 exports without losing a positive lower bound to Float64 underflow.
function enrich_lower_bound_row(r)
    r.omitted_tail_bound === missing && return merge(r, (;
        log_bubble_share_lower=missing, log10_bubble_share_lower=missing,
        bubble_share_lower_scientific=missing,
        lower_bound_float64_underflow=missing))
    log_lower = log(r.finite_prefix_product) - r.omitted_tail_bound
    log10_lower = log_lower / log(10.0)
    exponent10 = floor(Int, log10_lower)
    mantissa10 = 10.0^(log10_lower - exponent10)
    lower_scientific = @sprintf("%.8fe%+d", mantissa10, exponent10)
    underflow = log_lower < log(floatmin(Float64))
    return merge(r, (; log_bubble_share_lower=log_lower,
        log10_bubble_share_lower=log10_lower,
        bubble_share_lower_scientific=lower_scientific,
        lower_bound_float64_underflow=underflow))
end

theoretical_bubble_share_rows = enrich_lower_bound_row.(theoretical_bubble_share_rows)
selected_theoretical_bubble_rows = filter(
    r -> r.T_report == SELECTED_T_REPORT &&
         r.terminal_rule == selected_horizon_case.summary.terminal_rule,
    theoretical_bubble_share_rows)
bound_horizon_rows = filter(
    r -> r.evaluation_date && r.terminal_rule == "local_loglinear",
    theoretical_bubble_share_rows)
bubble_bound_cols = [
    "T_report", "model_T", "terminal_rule", "gate_source_terminal_rule",
    "cutoff_t", "cutoff_model_t", "t", "model_t", "evaluation_date",
    "bound_status", "lower_bound_certified", "hkt_gate_passed", "prefix_stable",
    "finite_prefix_product", "omitted_tail_bound",
    "bubble_share_lower", "log_bubble_share_lower",
    "log10_bubble_share_lower", "bubble_share_lower_scientific",
    "lower_bound_float64_underflow", "bubble_share_upper",
    "relative_interval_width_bound", "leakage", "dividend_leakage",
    "switch_leakage", "dividend_envelope", "switch_envelope",
    "Q_US", "NFA", "B_agg_lower", "B_agg_upper",
    "V_agg_lower", "V_agg_upper", "NFA_bubble_lower",
    "NFA_bubble_upper", "NFA_fund_lower", "NFA_fund_upper"]
write_csv(joinpath(OUTDIR, "theoretical_bubble_share_bounds.csv"),
          theoretical_bubble_share_rows, bubble_bound_cols)
write_csv(joinpath(OUTDIR, "theoretical_bubble_share_horizon_convergence.csv"),
          bound_horizon_rows, bubble_bound_cols)

# A failed finite-envelope diagnostic is a strictly worse outcome than a failed
# HKT gate and must not be reported as the latter.
function final_bound_status(rows)
    any(r -> r.bound_status == "invalid_finite_envelope_diagnostics", rows) &&
        return "INVALID: FINITE-ENVELOPE DIAGNOSTICS FAILED"
    all(r -> r.lower_bound_certified, rows) && return "CERTIFIED"
    all(r -> !r.hkt_gate_passed, rows) &&
        return "CONDITIONAL (HKT GATE FAILED: DIAGNOSTIC)"
    all(r -> r.hkt_gate_passed, rows) && return "CONDITIONAL ON FUTURE ENVELOPE"
    return "MIXED CERTIFICATION STATUS"
end

function plot_theoretical_bubble_bounds_final(rows; filename::String)
    tt = [r.t for r in rows]
    upper = Float64[r.bubble_share_upper for r in rows]
    lower = Float64[r.bubble_share_lower === missing ? NaN : r.bubble_share_lower for r in rows]
    log10_lower = Float64[r.log10_bubble_share_lower === missing ? NaN : r.log10_bubble_share_lower for r in rows]
    subtitle = "$(final_bound_status(rows)); cutoff K=$(first(rows).cutoff_t), solved T=$(first(rows).T_report)"
    p1 = plot(tt, upper, lw=2.4, color=:navy, label="upper: exact finite prefix",
        title="U.S. bubble-share interval\n" * subtitle,
        xlabel="notebook period t (model date t-1)", ylabel="B_US,t / q_US,t")
    plot!(p1, tt, lower, lw=2.2, ls=:dash, color=:darkorange,
        label="lower (Float64; see log panel)")
    p2 = plot(tt, log10_lower, lw=2.4, color=:darkorange,
        title="Positive theoretical lower bound (underflow-safe)",
        xlabel="notebook period t (model date t-1)",
        ylabel="log10 lower bound", label="log10 lower")
    widths = upper .- lower
    p3 = plot(tt, widths, lw=2.2, color=:purple,
        title="Absolute interval width",
        xlabel="notebook period t (model date t-1)",
        ylabel="upper - lower", label="width")
    fig = plot(p1, p2, p3, layout=(3, 1), size=(1040, 1040), bottom_margin=8mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

function plot_bound_horizon_convergence_final(rows; filename::String)
    horizons = sort(unique(r.T_report for r in rows))
    length(horizons) < 2 && return Markdown.parse(
        "_Only horizon(s) T=$(join(horizons, ", ")) are long enough to carry the " *
        "full bound at K=$(BOUND_CUTOFF); see the prefix-product stability panel instead._")
    p = plot(title="$(final_bound_status(rows)) bubble-share intervals across solve horizons\nshared pre-terminal cutoff K=$(BOUND_CUTOFF)",
        xlabel="no-buffer solve horizon T", ylabel="bubble share",
        legend=:outerright)
    for t in sort(unique(r.t for r in rows))
        rr = sort(filter(r -> r.t == t, rows); by=r -> r.T_report)
        Tvals = [r.T_report for r in rr]
        plot!(p, Tvals, [r.bubble_share_upper for r in rr],
              marker=:circle, lw=1.8, label="upper, t=$t")
        plot!(p, Tvals, [r.bubble_share_lower === missing ? NaN : r.bubble_share_lower for r in rr],
              marker=:diamond, ls=:dash, lw=1.8, label="lower, t=$t")
    end
    fig = plot(p, size=(1120, 640), bottom_margin=8mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

# With a short endpoint guard the reported prefix product is the object most
# exposed to terminal-closure error, so show it directly against every shorter
# hard-valid horizon that reaches the cutoff.
function plot_prefix_product_stability(all_rows; filename::String)
    # Drop the reference-against-itself row: it is identically zero, carries no
    # information, and on a log axis it stretches the panel over empty decades.
    rows = filter(r -> r.comparison == "vs_selected" &&
                       r.T_report != r.reference_T_report, all_rows)
    isempty(rows) && return Markdown.parse(
        "_No shorter horizon reaches K=$(BOUND_CUTOFF) for a stability comparison._")
    rr = sort(rows; by=r -> r.T_report)
    Tvals = [r.T_report for r in rr]
    logsafe(v) = [max(Float64(x), 1e-18) for x in v]
    p1 = plot(Tvals, logsafe([r.prefix_product_log_gap for r in rr]), marker=:circle,
        lw=2.2, color=:navy, yscale=:log10,
        title="Terminal-closure exposure of the reported prefix product\ncutoff K=$(BOUND_CUTOFF), reference T=$(SELECTED_T_REPORT)",
        xlabel="no-buffer solve horizon T reaching K",
        ylabel="|log P_{1,K} gap vs reference|", label="prefix-product log gap")
    hline!(p1, [BOUND_PREFIX_STABILITY_TOL], lc=:black, ls=:dash, lw=1.2,
        label="stability tolerance")
    p2 = plot(Tvals, logsafe([r.max_leakage_gap for r in rr]), marker=:square, lw=2.0,
        yscale=:log10, title="Pointwise prefix inputs",
        xlabel="no-buffer solve horizon T reaching K",
        ylabel="max gap over t=1:K", label="leakage a_t")
    plot!(p2, Tvals, logsafe([r.max_log_q_US_gap for r in rr]), marker=:diamond, lw=2.0,
        label="log q_US")
    plot!(p2, Tvals, logsafe([r.max_log_phi_US_gap for r in rr]), marker=:utriangle, lw=2.0,
        label="log phi_US")
    hline!(p2, [BOUND_PREFIX_STABILITY_TOL], lc=:black, ls=:dash, lw=1.2,
        label="stability tolerance")
    fig = plot(p1, p2, layout=(2, 1), size=(1040, 800), bottom_margin=8mm)
    savefig(fig, joinpath(OUTDIR, filename))
    return fig
end

display(plot_theoretical_bubble_bounds_final(
    selected_theoretical_bubble_rows; filename="theoretical_bubble_share_bounds.png"))
display(plot_bound_horizon_convergence_final(
    bound_horizon_rows; filename="theoretical_bubble_share_horizon_convergence.png"))
display(plot_prefix_product_stability(
    prefix_stability_rows; filename="bubble_bound_prefix_product_stability.png"))
display(markdown_table(
    filter(r -> r.evaluation_date, selected_theoretical_bubble_rows),
    ["t", "bound_status", "bubble_share_lower", "bubble_share_lower_scientific",
     "log10_bubble_share_lower", "lower_bound_float64_underflow",
     "bubble_share_upper", "relative_interval_width_bound"];
    maxrows=length(BOUND_EVALUATION_DATES)))

# Final reproducibility manifests: every resolved ProductionParams field,
# every effective run control, and hashes of all code/data inputs.
sha256_file(path) = bytes2hex(sha256(read(path)))
NOTEBOOK_FILE = joinpath(PROJECT_DIR, ZERO_NU_B_DIRNAME,
    "02_v9_ahp_hkt_scalar_tail_bubble_bounds_zero_nu_b.ipynb")
NB15_SOURCE_FILE = joinpath(PROJECT_DIR, COMMON_GROWTH_SOURCE_DIRNAME,
    "15_v9_ahp_pattern_calibration_search.jl")
NB16_SOURCE_FILE = joinpath(PROJECT_DIR, COMMON_GROWTH_SOURCE_DIRNAME,
    "16_v9_ahp_calibration_switch_at_16.jl")
notebook_code_source = read(
    `jupyter nbconvert --to script --stdout $NOTEBOOK_FILE`, String)
NOTEBOOK_CODE_SHA256 = bytes2hex(sha256(Vector{UInt8}(codeunits(notebook_code_source))))
p_configured_ref = tail_combo_params(
    T_max=ahp_reference_case.summary.T_report,
    n_buffer=ahp_reference_case.summary.n_buffer,
    branch_iters=ahp_reference_case.summary.branch_iters)
p_configured_selected = tail_combo_params(
    T_max=selected_horizon_case.summary.T_report,
    n_buffer=selected_horizon_case.summary.n_buffer,
    branch_iters=selected_horizon_case.summary.branch_iters)
p_default = ProductionParams()
override_fields = Set([:β, :γ, :π_persist, :a_US, :ϑ_US, :a_W, :H_W, :L_W,
    :A_X_US_u, :A_L_US_u, :ν_b, :ν_u, :ξ_u, :ξ_W, :ω̄, :ω̄_star,
    :κ, :χ, :η, :common_world_growth])
run_control_fields = Set([:T_max, :n_buffer, :branch_iters, :do_global_polish])
function parameter_origin(field)
    field == :ν_b && return "fixed_primitive_user_instruction"
    field == :common_world_growth && return "inert_compatibility_metadata"
    field in override_fields && return "notebook_16_override"
    field in run_control_fields && return "run_control"
    return "inherited_ProductionParams_default"
end
calibration_manifest_rows = NamedTuple[(;
    case_scope="global", parameter="selected_label", configured=AHP_SELECTED_LABEL,
    resolved=AHP_SELECTED_LABEL, default_value=missing,
    origin="notebook_15_winner",
    source="Notebook 15 candidate_summary.csv")]
parameter_case_specs = [
    (; case_scope="ahp_reference", configured=p_configured_ref, resolved=p_ref),
    (; case_scope="selected_downstream", configured=p_configured_selected,
       resolved=selected_horizon_case.result.params)]
for spec in parameter_case_specs
    for field in fieldnames(ProductionParams)
        origin = parameter_origin(field)
        source = field == :ν_b ?
            "primitive nu_b=0; no exponent unknown or runtime calibration" :
            field == :common_world_growth ?
            "inert compatibility metadata; omitted from the zero-exponent solver" :
            origin == "notebook_16_override" ? "Notebook 16 factory" :
            origin == "run_control" ?
                "Notebook 17 effective $(spec.case_scope) run control" :
            "TwoCountryProductionOLG.ProductionParams default"
        push!(calibration_manifest_rows, (; case_scope=spec.case_scope,
            parameter=String(field), configured=getfield(spec.configured, field),
            resolved=getfield(spec.resolved, field),
            default_value=getfield(p_default, field), origin, source))
    end
end
push!(calibration_manifest_rows, (; case_scope="fixed_counterfactual",
    parameter="nu_b_fixed",
    configured=ZERO_NU_B, resolved=p_ref.ν_b,
    default_value=p_default.ν_b, origin="fixed_primitive_user_instruction",
    source="zero-nu_b replication"))
push!(calibration_manifest_rows, (; case_scope="source_benchmark",
    parameter="nu_b_common_growth_reference",
    configured=AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE,
    resolved=AHP_SOURCE_COMMON_GROWTH_NU_B_REFERENCE,
    default_value=missing, origin="source_notebook_15_calibrated_output",
    source="source Notebook 15 candidate_summary.csv"))
write_csv(joinpath(OUTDIR, "calibration_manifest.csv"),
    calibration_manifest_rows,
    ["case_scope", "parameter", "configured", "resolved",
     "default_value", "origin", "source"])

artifact_scope = SEARCH_RIGHT_CENSORED ?
    "bounded_grid_right_censored_not_a_frontier" :
    "bounded_grid_with_unsafe_cap_within_iteration_budget"
selected_bound_summary = only(filter(
    r -> r.T_report == SELECTED_T_REPORT &&
         r.terminal_rule == selected_horizon_case.summary.terminal_rule,
    theoretical_tail_summary_rows))
run_manifest_rows = [
    (; field="run_started_utc", value=RUN_STARTED_UTC),
    (; field="artifact_scope", value=artifact_scope),
    (; field="run_mode", value=String(RUN_MODE)),
    (; field="project_dir", value=PROJECT_DIR),
    (; field="output_dir", value=OUTDIR),
    (; field="julia_version", value=string(VERSION)),
    (; field="machine", value=Sys.MACHINE),
    (; field="horizon_candidates", value=int_list_literal(HORIZON_T_CANDIDATES)),
    (; field="custom_horizon_grid", value=CUSTOM_HORIZON_GRID),
    (; field="configured_search_cap", value=CONFIGURED_SEARCH_CAP),
    (; field="horizon_search_cap", value=HORIZON_SEARCH_CAP),
    (; field="search_coarse_step", value=SEARCH_COARSE_STEP),
    (; field="refine_local_frontier", value=REFINE_LOCAL_FRONTIER),
    (; field="selected_T_report", value=SELECTED_T_REPORT),
    (; field="search_right_censored", value=SEARCH_RIGHT_CENSORED),
    (; field="search_outcome", value=SEARCH_OUTCOME),
    (; field="branch_iter_schedule", value=int_list_literal(BRANCH_ITER_SCHEDULE)),
    (; field="verify_selected_cold", value=VERIFY_SELECTED_COLD),
    (; field="cold_verification_attempt_note", value=COLD_VERIFICATION_ATTEMPT_NOTE),
    (; field="selected_cold_status", value=selected_horizon_cold_verification.cold_verification_status),
    (; field="selected_cold_max_log_phi_US_gap", value=selected_horizon_cold_verification.cold_vs_selected_max_log_phi_US_gap),
    (; field="selected_cold_max_log_q_US_gap", value=selected_horizon_cold_verification.cold_vs_selected_max_log_q_US_gap),
    (; field="reference_branch_iters", value=AHP_REFERENCE_BRANCH_ITERS),
    (; field="verify_buffered_reference", value=VERIFY_AHP_BUFFERED_REFERENCE),
    (; field="run_extended_terminal_rules", value=RUN_EXTENDED_TERMINAL_RULES),
    (; field="terminal_rules", value=join(String.(TERMINAL_RULES), ",")),
    (; field="ahp_match_tolerance", value=AHP_MATCH_TOL),
    (; field="residual_tolerance", value=RESID_TOL),
    (; field="psi_interior_tolerance", value=PSI_INTERIOR_TOL),
    (; field="equity_interior_tolerance", value=EQUITY_INTERIOR_TOL),
    (; field="theta_interior_tolerance", value=THETA_INTERIOR_TOL),
    (; field="phi_interior_margin", value=PHI_INTERIOR_MARGIN),
    (; field="accounting_scaled_tolerance", value=ACCOUNTING_SCALED_TOL),
    (; field="absorbing_invariance_tolerance", value=ABSORBING_INVARIANCE_TOL),
    (; field="cold_path_tolerance", value=COLD_PATH_TOL),
    (; field="hkt_e_ratio_tolerance", value=HKT_E_RATIO_TOL),
    (; field="hkt_qw_income_ratio_tolerance_diagnostic", value=HKT_QW_INCOME_RATIO_TOL),
    (; field="hkt_funding_gap_tolerance", value=HKT_FUNDING_LIMIT_GAP_TOL),
    (; field="hkt_successor_phi_gap_tolerance", value=HKT_SUCCESSOR_PHI_LOG_GAP_TOL),
    (; field="hkt_scalar_root_tolerance", value=HKT_SCALAR_ROOT_TOL),
    (; field="hkt_prerequisite_tail_window", value=HKT_PREREQ_TAIL_WINDOW),
    (; field="tail_regression_window", value=TAIL_REGRESSION_K[]),
    (; field="hkt_gate_passed_selected", value=HKT_GATE_PASSED),
    (; field="bound_evaluation_dates", value=int_list_literal(BOUND_EVALUATION_DATES)),
    (; field="bound_endpoint_guard_setting", value=BOUND_ENDPOINT_GUARD),
    (; field="configured_bound_cutoff", value=CONFIGURED_BOUND_CUTOFF),
    (; field="selected_solve_T", value=SELECTED_SOLVE_T),
    (; field="effective_bound_cutoff", value=BOUND_CUTOFF),
    (; field="effective_endpoint_guard", value=SELECTED_SOLVE_T - BOUND_CUTOFF),
    (; field="prefix_check_horizons",
       value=join(string.([c.summary.T_report for c in PREFIX_CHECK_CASES]), ";")),
    (; field="bound_capable_horizons",
       value=join(string.([c.summary.T_report for c in BOUND_CAPABLE_CASES]), ";")),
    (; field="max_prefix_product_log_gap",
       value=maximum(r.prefix_product_log_gap for r in prefix_stability_rows)),
    (; field="bound_prefix_comparison", value=selected_bound_summary.prefix_comparison),
    (; field="bound_prefix_reference_T_report", value=selected_bound_summary.prefix_reference_T_report),
    (; field="bound_prefix_product_log_gap", value=selected_bound_summary.prefix_product_log_gap),
    (; field="bound_prefix_max_log_q_US_gap", value=selected_bound_summary.prefix_max_log_q_US_gap),
    (; field="bound_prefix_product_stable", value=selected_bound_summary.prefix_stable),
    (; field="bound_prefix_paths_stable", value=selected_bound_summary.prefix_paths_stable),
    (; field="bound_prefix_stability_tolerance", value=BOUND_PREFIX_STABILITY_TOL),
    (; field="bound_cone_margin", value=BOUND_CONE_MARGIN),
    (; field="configured_zeta_lower", value=CONFIGURED_ZETA_LOWER),
    (; field="configured_q_price_epsilon", value=CONFIGURED_Q_PRICE_EPSILON),
    (; field="resolved_zeta_lower", value=selected_bound_summary.zeta_lower_assumption),
    (; field="resolved_zeta_lower_source", value=selected_bound_summary.zeta_lower_source),
    (; field="resolved_q_price_epsilon", value=selected_bound_summary.q_price_epsilon),
    (; field="resolved_q_price_epsilon_source", value=selected_bound_summary.q_price_epsilon_source),
    (; field="configured_lambda_ratio_bound", value=CONFIGURED_LAMBDA_BOUND),
    (; field="resolved_nu_b_uniform_bound", value=selected_bound_summary.nu_b_uniform_bound),
    (; field="resolved_nu_b_bound_source", value=selected_bound_summary.nu_b_bound_source),
    (; field="analytic_growth_floor_ok", value=selected_bound_summary.analytic_growth_floor_ok),
    (; field="omitted_tail_bound", value=selected_bound_summary.omitted_tail_bound),
    (; field="omitted_tail_numerical_estimate", value=selected_bound_summary.omitted_tail_numerical_estimate),
    (; field="envelope_slack_factor", value=selected_bound_summary.envelope_slack_factor),
    (; field="future_envelope_certified", value=FUTURE_ENVELOPE_CERTIFIED),
    (; field="model_file", value=MODEL_FILE),
    (; field="model_sha256", value=sha256_file(MODEL_FILE)),
    (; field="notebook_file", value=NOTEBOOK_FILE),
    (; field="notebook_input_sha256", value=sha256_file(NOTEBOOK_FILE)),
    (; field="notebook_hash_semantics", value="input notebook before nbconvert writes execution outputs"),
    (; field="notebook_code_sha256", value=NOTEBOOK_CODE_SHA256),
    (; field="notebook_code_hash_semantics", value="nbconvert Julia script; invariant to execution counts and outputs"),
    (; field="notebook_15_source_sha256", value=sha256_file(NB15_SOURCE_FILE)),
    (; field="notebook_16_source_sha256", value=sha256_file(NB16_SOURCE_FILE)),
    (; field="candidate_summary_sha256", value=sha256_file(NB15_CANDIDATE_CSV)),
    (; field="tracked_ahp_path_sha256", value=sha256_file(NB15_TRACKED_PATH_CSV)),
]
write_csv(joinpath(OUTDIR, "run_manifest.csv"),
          run_manifest_rows, ["field", "value"])

println("Saved output files in:")
println("  ", OUTDIR)
for f in sort(readdir(OUTDIR))
    println("  ", joinpath(OUTDIR, f))
end